# Capítulo 8: Regressão Linear

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 3 de James et al. (2023).

> De todos os princípios que se podem propor para esse fim, penso que não há nenhum mais geral, mais exato, nem de aplicação mais fácil do que aquele de que nos servimos nas pesquisas anteriores, e que consiste em tornar mínima a soma dos quadrados dos erros.
>
> — Adrien-Marie Legendre

Duzentos mercados gastaram em propaganda na TV, no rádio e no jornal, e venderam quantidades diferentes de um produto. O quanto o gasto ajuda a prever a venda? Este capítulo responde com o modelo paramétrico mais simples que existe, uma reta. A seção 8.1 ajusta essa reta por mínimos quadrados, à mão e com `LinearRegression`, e mostra que o critério RSS tem um único mínimo. A 8.2 separa a variação de vendas na parte que a reta explica e na que sobra no resíduo, e resume o ajuste em dois números, o RSE e o R², para dizer se aquela reta presta.

As seções seguintes soltam, uma de cada vez, as suposições que uma reta única esconde. A 8.3 junta as três mídias numa equação só e usa a correlação entre elas para explicar por que o coeficiente de jornal, positivo quando ajustado sozinho, encolhe para quase zero na companhia das outras duas mídias. O mesmo dado muda de resposta conforme o que entra na equação. A 8.4 passa aos dados `Credit` para tratar do preditor que não é número: a variável indicadora, a categoria-base e a leitura do intercepto como média de grupo.

A 8.5 testa duas suposições que sustentam tudo até ali: que os preditores agem em paralelo, e que cada um age em linha reta. Testa a primeira com termos de interação, entre `tv` e `radio` e entre `renda` e `estudante`; testa a segunda com um termo polinomial em `potencia` para prever `milhas_por_galao`, nos dados `Auto`. Nenhuma das duas extensões deixa de ser regressão linear. A soma continua linear nos coeficientes; só a coluna que cada um multiplica fica mais rica.

As duas últimas seções trocam de pergunta: não mais qual forma ajustar, mas o quanto confiar no ajuste. A 8.6 volta a `Auto` e a `Credit` para mostrar que o R² sozinho não denuncia um resíduo com padrão, um *outlier*, uma observação de alavancagem alta ou dois preditores colineares. Cada problema pede o seu instrumento para aparecer. A 8.7 compara a reta com o *k*-NN da seção 7.3 numa simulação em que a verdade é conhecida. A reta vence quando a verdade é dela mesma e perde conforme a curvatura cresce. Na simulação, ela volta a vencer assim que se soma um preditor sem relação com a resposta: é a maldição da dimensionalidade. Nenhum dos dois métodos vence sempre, e a escolha entre eles, sem conhecer a forma verdadeira de $f$, fica para o capítulo 10.

Ao final deste capítulo, você será capaz de:

- Ajustar uma reta por mínimos quadrados à mão (a partir de duas médias) e com `LinearRegression`, e ler nas curvas de nível de RSS o seu único mínimo
- Avaliar um ajuste de mínimos quadrados pelo RSE e pelo R², decompondo a variância da resposta na parcela que o modelo explica e na que sobra como resíduo, e reconhecer que o R² de treino nunca cai quando se acrescenta um preditor
- Ajustar um modelo com vários preditores ao mesmo tempo e explicar, pela correlação entre eles, por que o coeficiente de um muda de sinal ou de tamanho quando os demais entram na equação
- Codificar um preditor qualitativo em variáveis indicadoras, escolher a categoria-base e ler intercepto e coeficientes como médias de grupo
- Relaxar a suposição de aditividade com um termo de interação e a de linearidade com um termo polinomial, reconhecendo as duas extensões como regressão linear nos coeficientes
- Diagnosticar não linearidade, *outlier*, alavancagem e colinearidade com o instrumento apropriado a cada um (resíduo contra previsto, resíduo padronizado contra alavancagem, e o VIF, que é um número e não um gráfico) em vez de confiar só no R²
- Comparar o erro de teste da regressão linear com o do *k*-NN e reconhecer que a vantagem de um sobre o outro depende da forma verdadeira de $f$ e do número de preditores

## Seções

| Seção | Tópico |
|---|---|
| [8.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/01-regressao-linear-simples.html) | Regressão Linear Simples |
| [8.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/02-avaliando-o-ajuste.html) | Avaliando o Ajuste: R² e Erro |
| [8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-regressao-multipla.html) | Regressão Múltipla |
| [8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-preditores-qualitativos.html) | Preditores Qualitativos |
| [8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-interacao-e-termos-nao-lineares.html) | Interação e Termos Não Lineares |
| [8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-outliers-alavancagem-e-colinearidade.html) | *Outliers*, Alavancagem e Colinearidade |
| [8.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/07-regressao-linear-contra-k-vizinhos.html) | Regressão Linear contra *k*-NN |

## Regressão Linear Simples

> **📌 Nota**
>
> Esta seção corresponde às seções 3.1 e 3.1.1 de James et al. (2023).

`Advertising` traz o quanto duzentos mercados investiram em três mídias de propaganda (televisão, rádio e jornal) e quantas unidades do produto cada um vendeu. A pergunta mais simples que esse dado permite fazer é também a primeira: o investimento em TV, sozinho, ajuda a prever vendas? A regressão linear simples responde ajustando uma reta a exatamente dois números por mercado, `tv` e `vendas`, e deixa `radio` e `jornal` de fora por enquanto.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

### Uma reta para tv e vendas

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.shape, propaganda.columns.tolist()

Duzentos mercados e quatro colunas: `tv`, `radio` e `jornal` em milhares de dólares, `vendas` em milhares de unidades. Antes de qualquer modelo, a nuvem de `vendas` contra `tv`:

In [ ]:
# Figura: Vendas contra o investimento em TV nos duzentos mercados de Advertising.
fig, ax = plt.subplots()
ax.scatter(propaganda["tv"], propaganda["vendas"], color="C0", s=18)
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("vendas (milhares de unidades)")
plt.tight_layout()
plt.show()

Os pontos sobem juntos, mas não em fila: mercados com o mesmo gasto em TV vendem quantidades bem diferentes. Uma reta resume a tendência e deixa essa dispersão de fora. A regressão linear simples escreve essa reta como

$$
\text{vendas} \approx \beta_0 + \beta_1 \cdot \text{tv}.
$$

$\beta_0$ e $\beta_1$ são duas constantes, desconhecidas até se estimar. $\beta_0$ é o **intercepto**, o valor esperado de vendas quando `tv`, o investimento em TV, é zero. $\beta_1$ é a **inclinação**, o quanto `vendas` muda, em média, para cada unidade a mais de `tv`. O símbolo "≈" marca que a relação é uma aproximação: nada garante que dois mercados com o mesmo investimento em TV vendam exatamente o mesmo, e o quanto cada um foge da reta é o que o resto desta seção mede.

### O resíduo, e a soma que ele eleva ao quadrado

Uma vez que se tem estimativas $\hat\beta_0$ e $\hat\beta_1$, a reta prevê

$$
\hat y_i = \hat\beta_0 + \hat\beta_1 x_i,
$$

e o **resíduo** daquele mercado é a distância entre o que ele de fato vendeu e o que a reta previu para o mesmo investimento em TV:

$$
e_i = y_i - \hat y_i.
$$

A **soma dos quadrados dos resíduos** (RSS) soma esse erro, ao quadrado, sobre os duzentos mercados:

$$
\text{RSS} = e_1^2 + e_2^2 + \cdots + e_n^2 = \sum_{i=1}^{n} \left(y_i - \hat\beta_0 - \hat\beta_1 x_i\right)^2.
$$

Elevar ao quadrado, em vez de somar o valor absoluto de cada resíduo, pune um erro grande desproporcionalmente mais do que vários erros pequenos. E deixa RSS como uma soma de parábolas em $\beta_0$ e $\beta_1$: uma superfície com um único fundo, que se acha por fórmula fechada em vez de busca. Ajustar a reta é escolher, entre todos os pares $(\beta_0, \beta_1)$ possíveis, o único que minimiza essa soma: é a esse critério que se dá o nome de **mínimos quadrados**. Qual reta deixa a soma menor, olhando a nuvem acima? A que passa pelo meio dela, com inclinação positiva, é o palpite natural; a conta diz exatamente qual.

> **🔷 Conceito**
>
> O **resíduo** $e_i = y_i - \hat y_i$ mede a distância entre um valor observado e o previsto pela reta. A **soma dos quadrados dos resíduos** (RSS) soma $e_i^2$ sobre todas as observações. **Mínimos quadrados** é o critério que escolhe $\hat\beta_0$ e $\hat\beta_1$ minimizando RSS: nenhum outro par de coeficientes produz uma reta com RSS menor.

### A conta à mão: duas médias bastam

Minimizar RSS por cálculo, derivando em relação a $\beta_0$ e a $\beta_1$ e igualando as duas derivadas a zero, leva a uma fórmula fechada que depende só das médias de `tv` e `vendas`, e dos desvios de cada ponto em relação a elas:

$$
\hat\beta_1 = \frac{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)(y_i - \bar y)}{\displaystyle\sum_{i=1}^{n} (x_i - \bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x.
$$

Não é preciso nenhuma biblioteca de otimização. Dá para calcular direto com `pandas`, coluna contra coluna:

In [ ]:
tv = propaganda["tv"]
vendas = propaganda["vendas"]

tv_media = float(tv.mean())
vendas_media = float(vendas.mean())
beta1_mao = float(
    ((tv - tv_media) * (vendas - vendas_media)).sum() / ((tv - tv_media) ** 2).sum()
)
beta0_mao = vendas_media - beta1_mao * tv_media

(
    round(tv_media, 2),
    round(vendas_media, 2),
    round(beta1_mao, 4),
    round(beta0_mao, 4),
    round(beta1_mao * 1000, 1),
)

O investimento médio em TV é 147,04 (mil dólares); a venda média, 14,02 (mil unidades). A partir só dessas duas médias e dos desvios em relação a elas, $\hat\beta_1$ sai 0,0475 e $\hat\beta_0$, 7,0326. Como `tv` e `vendas` vêm as duas em milhares, $\hat\beta_1 \times 1.000$ traduz a inclinação para unidades vendidas por mil dólares: 47,5. Cada mil dólares a mais investidos em TV está associado, em média, a 47,5 unidades a mais vendidas.

### A mesma conta, pronta: `LinearRegression`

O `scikit-learn` resolve a mesma minimização por um caminho que vale para qualquer número de preditores, sem a fórmula das duas médias. `LinearRegression().fit(X, y)` recebe o preditor e a resposta e devolve um objeto já ajustado, com a inclinação em `.coef_` e o intercepto em `.intercept_`. Os dois vêm como array e escalar do `numpy`, por isso o `float(...)` ao redor de cada um daqui em diante. `X` entra como o `DataFrame` que já veio do `pandas`, mesmo sendo de uma coluna só, sem nenhum `.to_numpy()`: o estimador aceita e devolve, em `.feature_names_in_`, o nome de coluna que recebeu.

In [ ]:
X = propaganda[["tv"]]
y = propaganda["vendas"]

modelo = LinearRegression().fit(X, y)
beta1_sklearn = float(modelo.coef_[0])
beta0_sklearn = float(modelo.intercept_)

(
    modelo.feature_names_in_,
    (round(beta0_mao, 4), round(beta1_mao, 4)),
    (round(beta0_sklearn, 4), round(beta1_sklearn, 4)),
    bool(np.allclose([beta0_mao, beta1_mao], [beta0_sklearn, beta1_sklearn])),
)

> **🔧 Função**
>
> **`LinearRegression().fit(X, y)`**: cria um modelo linear e estima seus parâmetros por mínimos quadrados, a partir dos preditores `X` (uma tabela, uma coluna por preditor) e da resposta `y`. Devolve o próprio modelo, já ajustado.
>
> **`modelo.coef_`** e **`modelo.intercept_`**: os coeficientes estimados $\hat\beta_1, \dots, \hat\beta_p$, na ordem das colunas de `X`, e o intercepto $\hat\beta_0$.
>
> **`modelo.feature_names_in_`**: os nomes das colunas de `X` no momento do `fit`, na mesma ordem de `coef_`. Só existe quando `X` é um `DataFrame`.
>
> **`np.allclose(a, b)`**: `True` se cada elemento de `a` coincide com o correspondente de `b` até uma tolerância pequena, que absorve o último dígito de arredondamento do ponto flutuante.

`feature_names_in_` guarda só `tv`, o único nome que o `DataFrame` de uma coluna carregava. O par calculado à mão e o que saiu do `.fit()` são (7,0326; 0,0475) nos dois casos, e `np.allclose` confirma: `True`. As duas contas chegam ao mesmo mínimo de RSS, que é único, por caminhos diferentes. A fórmula das duas médias só existe com um preditor; o `.fit()` resolve o caso geral, e é ele que segue funcionando quando houver três preditores, na seção 8.3.

### A reta, e o resíduo que ela deixa

In [ ]:
# Figura: Duzentos mercados: vendas contra o investimento em TV, com a reta de mínimos quadrados por cima. Cada segmento verde liga um mercado observado à previsão da reta para o mesmo investimento: é o resíduo daquele mercado, o e_i que RSS eleva ao quadrado.
yhat = modelo.predict(X)
grade_tv = np.linspace(tv.min(), tv.max(), 200)
reta_grade = modelo.predict(pd.DataFrame({"tv": grade_tv}))

fig, ax = plt.subplots()
ax.vlines(
    tv, np.minimum(vendas, yhat), np.maximum(vendas, yhat), color="C2", linewidth=1
)
ax.plot(grade_tv, reta_grade, color="C1", linewidth=2, label="reta ajustada")
ax.scatter(tv, vendas, color="C0", s=18, zorder=3, label="observado")
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("vendas (milhares de unidades)")
ax.legend()
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`modelo.predict(X)`**: aplica o modelo ajustado a cada linha de `X` e devolve as previsões $\hat y$. `X` precisa ter as mesmas colunas, com os mesmos nomes, que o `fit` recebeu; por isso a grade entra como `pd.DataFrame({"tv": grade_tv})`.
>
> **`np.linspace(inicio, fim, n)`**: `n` números igualmente espaçados de `inicio` a `fim`, aqui os valores de `tv` onde a reta é desenhada.
>
> **`np.minimum(a, b)`** e **`np.maximum(a, b)`**: o menor e o maior de cada par de elementos. Cada segmento vertical vai do menor ao maior entre o valor observado e o previsto.

A reta captura a tendência de vender mais conforme se investe mais em TV, mas nenhum mercado senta exatamente sobre ela: sempre sobra um segmento. Somar o quadrado dos duzentos segmentos desta figura dá exatamente o RSS que a reta minimiza:

In [ ]:
rss_min = float(((vendas - yhat) ** 2).sum())
round(rss_min, 2)

O RSS da reta ajustada é 2.102,53. Nenhuma outra reta, entre todos os pares $(\beta_0, \beta_1)$ possíveis, soma menos que isso.

### Um vale com um fundo só

RSS, como soma de quadrados de uma função linear de $\beta_0$ e $\beta_1$, é uma superfície convexa nesses dois parâmetros. Desde que `tv` não seja a mesma em todos os mercados, a convexidade é estrita: um paraboloide elíptico, sem platôs e com um único ponto de mínimo. Variando $\beta_0$ e $\beta_1$ numa grade ao redor de $(\hat\beta_0, \hat\beta_1)$ e calculando RSS em cada combinação, essa forma aparece em curvas de nível, cada uma ligando os pares que produzem o mesmo RSS.

In [ ]:
# Figura: Curvas de nível de RSS sobre (β0, β1), na regressão de vendas sobre tv em Advertising, nos níveis 2.150, 2.200, 2.300, 2.400, 2.500 e 2.600. Quanto mais grossa a curva, menor o RSS que ela marca. O ponto laranja é (β̂0, β̂1), e as seis curvas se fecham ao redor dele.
grade_beta0 = np.linspace(3.5, 10.5, 300)
grade_beta1 = np.linspace(0.02, 0.075, 300)
malha_beta0, malha_beta1 = np.meshgrid(grade_beta0, grade_beta1)

tv_np = tv.to_numpy()
vendas_np = vendas.to_numpy()
residuo_grade = vendas_np - malha_beta0[:, :, None] - malha_beta1[:, :, None] * tv_np
rss_grade = (residuo_grade ** 2).sum(axis=2)

niveis_rss = np.array([2150.0, 2200.0, 2300.0, 2400.0, 2500.0, 2600.0])
larguras_das_curvas = [2.4, 2.0, 1.6, 1.3, 1.0, 0.7]

fig, ax = plt.subplots()
contornos = ax.contour(
    malha_beta0, malha_beta1, rss_grade,
    levels=niveis_rss, colors="C0", linewidths=larguras_das_curvas,
)
ax.scatter([beta0_sklearn], [beta1_sklearn], color="C1", s=40, zorder=3)
ax.set_xlabel(r"$\beta_0$")
ax.set_ylabel(r"$\beta_1$")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.meshgrid(a, b)`**: combina dois eixos numa grade, devolvendo duas matrizes com a coordenada horizontal e a vertical de cada ponto. Aqui, cada ponto da grade é um par $(\beta_0, \beta_1)$, e `rss_grade` guarda o RSS de cada par.
>
> **`ax.contour(x, y, z, levels, colors, linewidths)`**: desenha as curvas de nível de `z` sobre a grade, uma para cada valor de `levels`, com a espessura de `linewidths`.

A legenda promete curvas fechadas, e isso se confere contando, não olhando. O objeto que o `contour` devolve guarda, para cada nível, os segmentos de linha que desenhou, e um segmento fecha quando termina no mesmo ponto em que começou.

In [ ]:
segmentos_totais = 0
segmentos_fechados = 0
for segmentos_do_nivel in contornos.allsegs:
    for segmento in segmentos_do_nivel:
        if len(segmento) == 0:
            continue
        segmentos_totais += 1
        if np.allclose(segmento[0], segmento[-1]):
            segmentos_fechados += 1

segmentos_totais, segmentos_fechados, segmentos_fechados == segmentos_totais

> **🔧 Função**
>
> **`contornos.allsegs`**: a lista, nível por nível, das linhas que `ax.contour` desenhou. Cada linha é um array de pontos $(x, y)$; `segmento[0]` é o primeiro ponto e `segmento[-1]`, o último.

Seis níveis, seis segmentos desenhados, e os seis fecham: `segmentos_totais` e `segmentos_fechados` saem iguais, 6 e 6. Nenhuma curva sai cortada pela borda da janela. A figura é consistente com o vale único que a convexidade garante; a contagem só confirma que a janela escolhida mostra esse vale inteiro.

In [ ]:
minimo_real_menor_que_grade = bool(rss_min < rss_grade.min())
round(rss_min, 2), round(float(rss_grade.min()), 2), minimo_real_menor_que_grade

O mínimo verdadeiro, 2.102,53, fica abaixo até do menor valor que a própria grade alcança, 2.102,56, e `minimo_real_menor_que_grade` é `True`. Nenhum dos 300×300 pontos testados coincide exatamente com $(\hat\beta_0, \hat\beta_1)$, só passa perto. É a fórmula fechada, não a grade, que encontra o fundo do vale de verdade. RSS diz qual par de coeficientes é o melhor entre os que este dado observou; não diz se essa reta presta para prever vendas em geral, nem quanto de vendas ela de fato explica.

## Avaliando o Ajuste: R² e Erro

> **📌 Nota**
>
> Esta seção corresponde à seção 3.1.3 de James et al. (2023).

Mínimos quadrados sempre devolve um $\hat\beta_0$ e um $\hat\beta_1$, mesmo quando a reta descreve mal o dado. Para `vendas ~ tv`, quanto essa reta *serve*? Quanto um mercado costuma ficar longe dela, e quanto da variação de vendas entre os mercados ela dá conta de explicar? Duas medidas respondem, cada uma a uma dessas perguntas, e as duas partem do mesmo RSS.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use("estilo-figuras.mplstyle")

propaganda = pd.read_csv("dados/Advertising.csv")
X = propaganda[["tv"]]
y = propaganda["vendas"]
modelo = LinearRegression().fit(X, y)
yhat = modelo.predict(X)

### O erro típico de previsão: RSE

O erro-padrão residual (RSE) resume o RSS como um único número, na mesma unidade da resposta. É a raiz do RSS dividido por $n-2$ em vez de $n$, porque a reta já consumiu dois parâmetros, $\hat\beta_0$ e $\hat\beta_1$, antes de sobrar erro para medir:

$$
\text{RSE} = \sqrt{\frac{\text{RSS}}{n-2}}
$$

In [ ]:
n = len(y)
n_menos_2 = n - 2
rss = float(((y - yhat) ** 2).sum())
rse_mao = float(np.sqrt(rss / n_menos_2))
rse_mil_unidades = round(rse_mao * 1000)

mse_sklearn = mean_squared_error(y, yhat)
rse_via_mse = float(np.sqrt(mse_sklearn * n / n_menos_2))

vendas_media = float(y.mean())
percentual_erro = rse_mao / vendas_media * 100

(
    n,
    n_menos_2,
    round(rss, 2),
    round(rse_mao, 4),
    round(rse_via_mse, 4),
    bool(np.isclose(rse_mao, rse_via_mse)),
    rse_mil_unidades,
    round(vendas_media, 4),
    round(percentual_erro, 2),
)

> **🔧 Função**
>
> **`mean_squared_error(y, yhat)`**: a média de $(y_i - \hat y_i)^2$, isto é, o RSS dividido por $n$.

Duzentos mercados, `n_menos_2 = 198` depois de descontar os dois parâmetros que a reta gastou para se ajustar: RSS, 2.102,53, dividido por 198 e com a raiz, dá 3,2587. `mean_squared_error` divide o mesmo RSS por $n$, e não por $n-2$. Para recuperar o RSE a partir dele, é preciso desfazer essa divisão antes de tirar a raiz. Feito isso, os dois caminhos batem: 3,2587 nos dois casos, `True`.

`vendas` está em milhares de unidades, então o afastamento típico entre a venda de um mercado e a reta é da ordem de 3.259 unidades, para cima ou para baixo. "Típico" aqui tem sentido preciso: o RSE é a raiz de uma média de quadrados, e estima o desvio padrão do erro $\varepsilon$ em torno da reta. Contra a venda média de 14,0225 mil unidades, esses 3,2587 mil unidades representam 23,24% dela. É esse o tamanho do erro típico de previsão, relativo à própria escala do que se está prevendo.

Esse único número vale para a reta inteira, e só descreve bem cada mercado se a dispersão em torno da reta for parecida em toda a faixa de `tv`. A nuvem da seção 8.1 sugeria que não é: estreita perto de zero, aberta à direita. O mesmo "tamanho típico", a raiz da média dos resíduos ao quadrado, medido dentro de faixas de `tv`:

In [ ]:
residuo = y - yhat
faixa_tv = pd.cut(X["tv"], [0, 50, 100, 150, 200, 300])
residuo_tipico = pd.DataFrame({
    "mercados": residuo.groupby(faixa_tv, observed=True).size(),
    "resíduo típico": (residuo**2).groupby(faixa_tv, observed=True).mean() ** 0.5,
}).round(2)
residuo_tipico

In [ ]:
menor_tipico = float(residuo_tipico["resíduo típico"].min())
maior_tipico = float(residuo_tipico["resíduo típico"].max())
rse_fica_entre_as_faixas = bool(menor_tipico < rse_mao < maior_tipico)

menor_tipico, maior_tipico, rse_fica_entre_as_faixas

In [ ]:
# Figura: Resíduo de cada mercado contra o investimento em TV, no ajuste de vendas sobre tv. A faixa laranja vai de menos a mais o resíduo típico de cada faixa de tv; a linha tracejada verde marca ± RSE, o mesmo valor para todos os mercados.
bordas = [0, 50, 100, 150, 200, 300]
fig, ax = plt.subplots()
ax.axhline(0, color="#6C757D", linewidth=0.8)
for sinal in (1, -1):
    ax.axhline(sinal * rse_mao, color="C2", linewidth=1.2, linestyle="--")
ax.scatter(X["tv"], residuo, color="C0", s=14, alpha=0.6)
for esquerda, direita, tipico in zip(
    bordas[:-1], bordas[1:], residuo_tipico["resíduo típico"]
):
    ax.fill_between(
        [esquerda, direita], -tipico, tipico, color="C1", alpha=0.18, linewidth=0
    )
ax.set_xlim(0, 300)
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("resíduo (mil unidades)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`pd.cut(serie, bordas)`**: põe cada valor da série na faixa entre duas bordas consecutivas. Aqui, os mercados ficam agrupados por faixa de investimento em TV.
>
> **`ax.fill_between(x, y1, y2)`**: pinta a região entre as curvas `y1` e `y2` ao longo de `x`, aqui um retângulo por faixa de `tv`.

O resíduo típico vai de 1,78 mil unidades nos mercados que gastam até 50 mil dólares em TV a 4,39 nos que gastam mais de 200, e `rse_fica_entre_as_faixas` confirma que o RSE de 3,2587 fica entre esses extremos. O RSE é um resumo da reta inteira: para os mercados que gastam pouco em TV ele exagera o erro, e para os que gastam muito ele o subestima. Lê-lo como "o desvio padrão de $\varepsilon$" supõe uma dispersão constante que este dado não tem.

### R²: a proporção de variância explicada

RSE vem na unidade de `vendas`, e 3,2587 só diz alguma coisa a quem sabe a escala de vendas. Se vendas fosse medida em unidades em vez de milhares, o mesmo ajuste teria RSE mil vezes maior. Existe uma medida que não dependa da escala? R² descarta a unidade: é a fração do TSS (a variação total da resposta em torno da própria média, antes de qualquer reta) que a regressão explica.

$$
\text{TSS} = \sum_{i=1}^n (y_i - \bar y)^2, \qquad R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
$$

In [ ]:
tss = float(((y - vendas_media) ** 2).sum())
explicada = tss - rss

r2_mao = 1 - rss / tss
r2_sklearn = float(r2_score(y, yhat))
r2_via_score = float(modelo.score(X, y))

(
    round(tss, 2),
    round(explicada, 2),
    round(r2_mao, 4),
    round(r2_sklearn, 4),
    round(r2_via_score, 4),
    bool(np.isclose(r2_mao, r2_sklearn) and np.isclose(r2_mao, r2_via_score)),
    round(r2_mao * 100, 2),
)

> **🔧 Função**
>
> **`r2_score(y, yhat)`**: o R², $1 - \text{RSS}/\text{TSS}$, calculado a partir das respostas observadas e das previstas.
>
> **`modelo.score(X, y)`**: nos estimadores de regressão do `scikit-learn`, o mesmo R², com o `predict` feito por dentro.
>
> **`np.isclose(a, b)`**: a versão de `np.allclose` para dois números, `True` se coincidem até uma tolerância pequena.

TSS, a variação total de vendas antes de qualquer reta, sai 5.417,15. Descontado o RSS de 2.102,53, sobram 3.314,62 que a reta explica. A conta à mão, `r2_score` e `.score()` concordam em 0,6119: `True`. Uma reta que usa só o investimento em TV explica 61,19% da variância de vendas, e esse número não carrega unidade nenhuma: continuaria 0,6119 com vendas medida em unidades, em milhares ou em milhões.

> **🔷 Conceito**
>
> O **erro-padrão residual** (RSE) é a raiz de $\text{RSS}/(n-2)$: o tamanho típico do resíduo, na unidade da resposta. O **R²** é $1 - \text{RSS}/\text{TSS}$: a fração da variância da resposta que o modelo explica nos dados usados no ajuste. Nesses dados, e com intercepto, fica entre 0 e 1, não tem unidade e não muda se a resposta trocar de escala.
>
> Os dois são medidas de **treino**. Acrescentar um preditor a um modelo nunca faz o R² de treino cair, mesmo que o preditor seja ruído puro: no pior caso, mínimos quadrados lhe dá coeficiente zero e o RSS fica onde estava. Comparar o R² de um modelo com o de outro que o contém, portanto, sempre favorece o maior. A comparação honesta é o erro em dado novo, o MSE de teste da seção 7.6.

R² alto não garante que o modelo seja bom, nem R² baixo que ele seja ruim. Os dois dependem de quanto do problema é o $\mathrm{Var}(\varepsilon)$, o piso irredutível da seção 7.6: nenhuma reta, nem a melhor possível, explica a parte da variância que é ruído puro.

### O que a reta previu contra o que cada mercado vendeu

In [ ]:
soma_confere = bool(np.isclose(rss + explicada, tss))
explicada_maior_que_rss = bool(explicada > rss)
soma_confere, explicada_maior_que_rss

RSS mais a parte explicada soma de volta o TSS (`soma_confere` é `True`), e a parte explicada é maior que a não explicada (`explicada_maior_que_rss` também é `True`). É o mesmo fato que R² > 0,5 já dizia.

In [ ]:
# Figura: À esquerda, previsto contra observado para os duzentos mercados de Advertising, com a diagonal de previsão perfeita: a distância vertical de cada ponto até ela é o resíduo daquele mercado. À direita, o TSS decomposto em RSS (não explicada) e a parte que a reta explica, com R² anotado como a fração de cima.
fig, (ax_diag, ax_barra) = plt.subplots(1, 2, figsize=(9, 4.2))

limite = (min(y.min(), yhat.min()) - 1, max(y.max(), yhat.max()) + 1)
ax_diag.plot(limite, limite, color="C1", linewidth=1.5, label="previsão perfeita")
ax_diag.scatter(y, yhat, color="C0", s=16, alpha=0.7, label="mercado")
ax_diag.set_xlim(limite)
ax_diag.set_ylim(limite)
ax_diag.set_aspect("equal")
ax_diag.set_xlabel("vendas observadas (mil unidades)")
ax_diag.set_ylabel("vendas previstas (mil unidades)")
ax_diag.legend()

ax_barra.bar(0, rss, color="#9AA5B1", label="RSS (não explicada)")
ax_barra.bar(0, explicada, bottom=rss, color="C1", label="explicada (TSS − RSS)")
ax_barra.annotate(
    f"R² = {r2_mao:.3f}".replace(".", ","),
    xy=(0, rss + explicada / 2),
    xytext=(0.55, rss + explicada / 2),
    va="center",
    fontsize=9,
    arrowprops={"arrowstyle": "-", "linewidth": 0.8, "color": "#6C757D"},
)
ax_barra.set_xlim(-0.6, 1.6)
ax_barra.set_xticks([0])
ax_barra.set_xticklabels(["TSS"])
ax_barra.set_ylabel("soma de quadrados")
ax_barra.legend(loc="upper right")

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.bar(x, altura, bottom)`**: desenha uma barra de altura `altura` na posição `x`, começando em `bottom` em vez de zero. Duas chamadas, a segunda com `bottom` igual à altura da primeira, empilham as duas parcelas numa barra só.

À esquerda, cada ponto é um mercado: quanto mais perto da diagonal, menor o resíduo que entra ao quadrado no RSS. À direita, a barra do TSS aparece dividida nas duas parcelas que R² compara. O pedaço cinza é o RSS que a reta deixou sem explicar; o laranja é a fatia que a reta de mínimos quadrados explica, e que R² = 0,612 mede como proporção do todo.

## Regressão Múltipla

> **📌 Nota**
>
> Esta seção corresponde à seção 3.2 de James et al. (2023).

`Advertising` traz três mídias, e a seção 8.1 usou só uma. O caminho mais direto seria ajustar uma reta para cada mídia, como a 8.1 fez com `tv`, e ler cada coeficiente isolado. Feito para `jornal`, esse caminho dá uma resposta que não sobrevive a um modelo com as três mídias juntas.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.style.use("estilo-figuras.mplstyle")

propaganda = pd.read_csv("dados/Advertising.csv")
y = propaganda["vendas"]

### Jornal, sozinho, parece importar

In [ ]:
X_jornal = propaganda[["jornal"]]
modelo_jornal = LinearRegression().fit(X_jornal, y)
beta1_jornal_sozinho = float(modelo_jornal.coef_[0])

round(beta1_jornal_sozinho, 4), round(beta1_jornal_sozinho * 1000, 1)

Ajustada sozinha contra `jornal`, a reta de mínimos quadrados dá um coeficiente de 0,0547: cada mil dólares a mais gastos em jornal está associado, em média, a 54,7 unidades a mais vendidas. É um coeficiente positivo, do mesmo tipo que a seção 8.1 encontrou para `tv`, e nada nesta regressão isolada distingue `jornal` das outras duas mídias.

Só que ajustar `jornal` sozinho ignora `tv` e `radio` por completo. Se as três mídias variam juntas de mercado para mercado, o coeficiente de uma pode estar levando crédito por vendas que outra produziu. A única forma de saber é colocar as três na mesma equação.

### As três mídias na mesma equação

O modelo múltiplo estende a mesma ideia de mínimos quadrados a mais de um preditor. Em vez de uma reta, ajusta um hiperplano que minimiza a soma dos quadrados dos resíduos sobre `tv`, `radio` e `jornal` ao mesmo tempo:

$$
\text{vendas} \approx \beta_0 + \beta_1 \cdot \text{tv} + \beta_2 \cdot \text{radio} + \beta_3 \cdot \text{jornal}.
$$

Antes de olhar a saída, vale apostar: o coeficiente de `jornal`, 0,0547 quando ajustado sozinho, fica parecido quando `tv` e `radio` entram na mesma equação?

In [ ]:
X_multiplo = propaganda[["tv", "radio", "jornal"]]
modelo_multiplo = LinearRegression().fit(X_multiplo, y)

coeficientes = pd.Series(modelo_multiplo.coef_, index=modelo_multiplo.feature_names_in_)
coeficientes_mil_dolares = (coeficientes * 1000).round(1)

coeficientes.round(4), coeficientes_mil_dolares

> **🔧 Função**
>
> **`pd.Series(valores, index=rotulos)`**: uma coluna de valores com um rótulo por posição. Aqui, os rótulos são os nomes de `feature_names_in_`, e cada coeficiente passa a ser lido pelo nome do seu preditor.

`feature_names_in_` guarda os três nomes de coluna que o `DataFrame` carregava, na mesma ordem dos coeficientes, e por isso dá para juntar os dois numa única `Series` em vez de decorar qual número é qual. `tv` sai com 0,0458 (45,8 unidades por mil dólares) e `radio` com 0,1885 (188,5 por mil dólares), os dois positivos. `jornal` sai com -0,0010 (-1,0 por mil dólares): praticamente zero, e de sinal oposto ao 0,0547 que a regressão sozinha tinha encontrado para ele.

> **🔷 Conceito**
>
> Num modelo múltiplo, cada coeficiente $\hat\beta_j$ se lê como o efeito médio sobre a resposta de aumentar o preditor $X_j$ em uma unidade, **mantendo todos os outros preditores fixos**. É essa cláusula que separa a leitura de um coeficiente múltiplo da de uma regressão simples, e é ela que muda o que se pode dizer sobre `jornal`.

Lido dessa forma: gastar mais mil dólares em `tv`, mantendo `radio` e `jornal` fixos, está associado a 45,8 unidades a mais de venda; mais mil dólares em `radio`, mantendo `tv` e `jornal` fixos, a 188,5 unidades a mais. Mais mil dólares em `jornal`, mantendo `tv` e `radio` fixos, está associado a uma variação de -1,0 unidade. Isso é pouco? Um coeficiente sozinho não diz, porque depende da unidade do preditor. O que diz é o quanto a previsão muda quando `jornal` percorre toda a faixa observada, comparado com o quanto as vendas variam:

In [ ]:
amplitude_jornal = float(propaganda["jornal"].max() - propaganda["jornal"].min())
efeito_jornal_na_faixa = float(coeficientes["jornal"]) * amplitude_jornal
desvio_vendas = float(y.std())

round(amplitude_jornal, 1), round(efeito_jornal_na_faixa, 3), round(desvio_vendas, 2)

Indo do mercado que menos gasta em jornal ao que mais gasta, uma faixa de 113,7 mil dólares, a previsão muda -0,118 mil unidades, isto é, 118 unidades a menos. `vendas` varia de um mercado para outro com desvio padrão de 5,22 mil unidades. Depois que `tv` e `radio` estão na conta, sobra a `jornal` um efeito praticamente nulo.

### Por que jornal some: quem anda com quem

A explicação não exige mais uma regressão. Basta ver como as três mídias se relacionam entre si, antes de qualquer venda entrar na conta.

In [ ]:
correlacoes = propaganda[["tv", "radio", "jornal", "vendas"]].corr()
correlacoes.round(4)

> **🔧 Função**
>
> **`df.corr()`**: a correlação de Pearson de cada coluna com cada outra, numa tabela quadrada. `correlacoes.loc["jornal", "radio"]` lê uma célula pelo nome da linha e da coluna.

In [ ]:
corr_jornal_radio = float(correlacoes.loc["jornal", "radio"])
corr_jornal_tv = float(correlacoes.loc["jornal", "tv"])
jornal_mais_correlacionado_com_radio = bool(corr_jornal_radio > corr_jornal_tv)

(
    round(corr_jornal_radio, 4),
    round(corr_jornal_tv, 4),
    jornal_mais_correlacionado_com_radio,
)

`jornal` correlaciona com `radio` a 0,3541 e com `tv` a só 0,0566, e `jornal_mais_correlacionado_com_radio` confirma que a primeira supera a segunda. Mercados que gastam mais em rádio tendem a gastar mais em jornal também: os dois investimentos sobem e descem juntos, sem que isso implique nada causal entre eles.

É essa correlação que explica a reviravolta do coeficiente de jornal, de 0,0547 sozinho para -0,0010 na companhia das outras duas mídias. Suponha que é o rádio, e não o jornal, que de fato move vendas. Então nos mercados onde se gasta mais em rádio as vendas tendem a ser maiores, e a tabela de correlação mostra que esses são os mesmos mercados que gastam mais em jornal. Uma regressão que olha só para `jornal`, sem `radio` por perto, não tem como separar as duas coisas e atribui a jornal parte do crédito que é do rádio. Colocar as duas mídias na mesma equação é o que permite distinguir, e é aí que o coeficiente de jornal cai a praticamente zero.

### R² e RSE: o quanto o modelo múltiplo melhora

R² não muda de definição com mais preditores; o RSE muda de denominador. Com $p$ preditores, a raiz é de $\text{RSS}/(n - p - 1)$, com um grau gasto pelo intercepto e um por coeficiente. O $n-2$ da seção 8.2 é o caso $p = 1$ dessa mesma forma.

In [ ]:
X_tv = propaganda[["tv"]]
modelo_tv = LinearRegression().fit(X_tv, y)
yhat_tv = modelo_tv.predict(X_tv)

n = len(y)
rss_tv = float(((y - yhat_tv) ** 2).sum())
rse_tv = float(np.sqrt(rss_tv / (n - 2)))
r2_tv = float(r2_score(y, yhat_tv))

round(rse_tv, 4), round(r2_tv, 4)

In [ ]:
yhat_multiplo = modelo_multiplo.predict(X_multiplo)

p = X_multiplo.shape[1]
rss_multiplo = float(((y - yhat_multiplo) ** 2).sum())
rse_multiplo = float(np.sqrt(rss_multiplo / (n - p - 1)))
r2_multiplo = float(r2_score(y, yhat_multiplo))

round(rse_multiplo, 4), round(r2_multiplo, 4)

In [ ]:
comparacao = pd.DataFrame(
    {"R²": [r2_tv, r2_multiplo], "RSE": [rse_tv, rse_multiplo]},
    index=["tv sozinho", "tv + radio + jornal"],
).round(4)
comparacao

O modelo de `tv` sozinho explica 0,6119 da variância de vendas, com erro típico de 3,2587 mil unidades. Somar `radio` e `jornal` sobe o R² para 0,8972 e derruba o RSE para 1,6855. Sobre os duzentos mercados usados no ajuste, as três mídias juntas erram menos, em média, do que `tv` sozinho.

Quanto desse ganho é mérito das duas mídias novas? O R² sozinho não responde: ele nunca cai quando se acrescenta um preditor, nem mesmo com uma coluna de ruído no lugar de `radio`. O que pesa aqui é o tamanho do salto, de 0,6119 para 0,8972, e o RSE, que desconta um grau por coeficiente, cair de 3,2587 para 1,6855, quase à metade.

### A superfície ajustada com tv e radio

Como jornal quase não muda a previsão, a superfície que os coeficientes de `tv` e `radio` desenham já carrega quase toda a informação do modelo múltiplo. E, com só dois preditores, dá para desenhar essa superfície inteira.

In [ ]:
X_tv_radio = propaganda[["tv", "radio"]]
modelo_tv_radio = LinearRegression().fit(X_tv_radio, y)
yhat_tv_radio = modelo_tv_radio.predict(X_tv_radio)

r2_tv_radio = float(r2_score(y, yhat_tv_radio))
coef_tv = float(modelo_tv_radio.coef_[0])
coef_radio = float(modelo_tv_radio.coef_[1])
round(r2_tv_radio, 4), round(coef_tv, 4), round(coef_radio, 4)

In [ ]:
# Figura: Vendas previstas para toda combinação de investimento em tv e radio, pelo modelo ajustado com as duas mídias. Os pontos são os duzentos mercados observados.
grade_tv = np.linspace(propaganda["tv"].min(), propaganda["tv"].max(), 60)
grade_radio = np.linspace(propaganda["radio"].min(), propaganda["radio"].max(), 60)
malha_tv, malha_radio = np.meshgrid(grade_tv, grade_radio)
grade = pd.DataFrame({"tv": malha_tv.ravel(), "radio": malha_radio.ravel()})
superficie = modelo_tv_radio.predict(grade).reshape(malha_tv.shape)

laranjas = LinearSegmentedColormap.from_list(
    "laranjas", plt.get_cmap("Oranges")(np.linspace(0.15, 0.70, 256))
)

fig, ax = plt.subplots()
mapa = ax.contourf(malha_tv, malha_radio, superficie, levels=10, cmap=laranjas)
ax.contour(
    malha_tv,
    malha_radio,
    superficie,
    levels=mapa.levels,
    colors="white",
    linewidths=0.6,
)
ax.scatter(
    propaganda["tv"],
    propaganda["radio"],
    color="C0",
    s=16,
    edgecolor="white",
    linewidth=0.5,
)
ax.set_xlabel("tv (milhares de dólares)")
ax.set_ylabel("radio (milhares de dólares)")
barra = fig.colorbar(mapa, ax=ax, shrink=0.9, pad=0.02)
barra.set_label("vendas previstas (mil unidades)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`LinearSegmentedColormap.from_list(nome, cores)`**: monta uma escala de cores contínua a partir de uma lista de cores. Aqui, as cores são uma fatia da escala `"Oranges"`, `plt.get_cmap("Oranges")(np.linspace(0.15, 0.70, 256))`, que evita os extremos quase brancos e quase pretos da escala.
>
> **`ax.contourf(x, y, z, levels, cmap)`**: pinta o plano por faixas de valor de `z`, como num mapa de relevo; `levels` é o número de faixas e `cmap`, a escala de cores. Por cima, o `ax.contour` da seção 8.1, com os mesmos `levels`, traça em branco a fronteira entre cada par de faixas vizinhas.

Cada faixa reúne as combinações de `tv` e `radio` que o modelo prevê com a mesma venda. As fronteiras entre faixas, em branco, são retas paralelas, porque o modelo é um plano. A superfície cresce para a direita e para cima, como anunciam os dois coeficientes positivos deste ajuste, 0,0458 para `tv` e 0,1880 para `radio`.

### O que a superfície plana ainda erra

O modelo com `tv` e `radio` explica 0,8972 da variância de vendas, o mesmo valor, até a quarta casa, do modelo de três mídias da tabela acima. `jornal` quase não muda a previsão. Mas sobra no resíduo um padrão que essa superfície plana não captura.

In [ ]:
df_resid = pd.DataFrame(
    {
        "previsto": yhat_tv_radio,
        "residuo": y.to_numpy() - yhat_tv_radio,
    }
)
df_resid["terco"] = pd.qcut(df_resid["previsto"], 3, labels=["baixo", "medio", "alto"])

medias_por_terco = df_resid.groupby("terco", observed=True)[
    ["previsto", "residuo"]
].mean()
medias_por_terco.round(4)

> **🔧 Função**
>
> **`pd.qcut(serie, q, labels)`**: corta os valores de `serie` em `q` faixas com o mesmo número de observações cada, pelos quantis, e dá a cada faixa um rótulo de `labels`. É o primo de `pd.cut`, que corta nas bordas que recebe.
>
> **`df.groupby(coluna, observed=True)[colunas].mean()`**: separa as linhas pelos valores de `coluna` e tira a média de cada uma das `colunas` dentro de cada grupo. `observed=True` mantém só os grupos que de fato aparecem, o comportamento que `pandas` recomenda para colunas categóricas.

Os duzentos mercados foram divididos em três grupos pelo valor previsto: o terço com a previsão mais baixa, o do meio e o mais alto. O resíduo médio não fica perto de zero nos três grupos: 0,4307 no terço mais baixo, -0,8077 no do meio, e de volta a 0,3650 no mais alto. O sinal do erro muda com a faixa de previsão, em vez de se espalhar ao acaso ao redor de zero.

In [ ]:
# Figura: Resíduo contra o valor previsto, no modelo de tv e radio. A linha tracejada marca resíduo zero; os três marcadores laranja são a média de cada terço de previsão, abaixo de zero no meio e acima nas duas pontas.
fig, ax = plt.subplots()
ax.axhline(0, color="C2", linewidth=1, linestyle="--")
ax.scatter(df_resid["previsto"], df_resid["residuo"], color="C0", s=14, alpha=0.6)
ax.plot(
    medias_por_terco["previsto"],
    medias_por_terco["residuo"],
    color="C1",
    linewidth=2,
    marker="o",
    markersize=7,
)
ax.set_xlabel("vendas previstas (mil unidades)")
ax.set_ylabel("resíduo (observada − prevista)")
plt.tight_layout()
plt.show()

Uma curvatura assim, negativa no meio da faixa de previsão e positiva nas duas pontas, diz que o plano erra de forma sistemática, mas não diz, sozinha, por quê. Ela é compatível com duas causas: o efeito de uma mídia depende do nível da outra, ou alguma mídia age sobre vendas em curva, e não em linha reta. A seção 8.5 testa a primeira hipótese com este mesmo modelo.

## Preditores Qualitativos

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.1 de James et al. (2023).

Um coeficiente de regressão multiplica um número por outro, $\beta_j \cdot x_j$. Um preditor como `tv`, em milhares de dólares, entra nessa conta sem cerimônia. `Credit` tem quatro colunas que não são números: `imovel_proprio`, `estudante` e `casado` valem `sim` ou `não`, e `regiao` vale `Leste`, `Sul` ou `Oeste`. Nenhuma delas multiplica coeficiente nenhum do jeito que está. Como fazer uma categoria entrar numa regressão?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
credit.shape, credit.select_dtypes("number").columns.tolist()

In [ ]:
{
    coluna: sorted(credit[coluna].unique())
    for coluna in credit.select_dtypes(exclude="number").columns
}

> **🔧 Função**
>
> **`df.select_dtypes("number")`**: as colunas numéricas de `df`; com `exclude="number"`, as demais. **`serie.unique()`**: os valores distintos da coluna.

São 400 clientes e onze colunas. Sete são numéricas: seis preditores (`renda`, `limite`, `pontuacao`, `cartoes`, `idade` e `escolaridade`) e `saldo`. Das quatro que não são números, três valem `sim` ou `não` (`imovel_proprio`, `estudante` e `casado`) e `regiao` tem três categorias. `saldo` é a dívida média no cartão de crédito de cada cliente, em dólares. É a resposta que o resto da seção tenta prever a partir de duas das colunas categóricas, `estudante` e `regiao`.

### O saldo médio, por ser estudante ou não

A pergunta mais simples é a de duas categorias: quem é estudante carrega, em média, saldo diferente de quem não é? Antes de qualquer modelo, o saldo de cada cliente, separado pelas duas categorias:

In [ ]:
# Figura: Saldo de cada um dos 400 clientes, separado por ser estudante ou não. Os pontos têm um deslocamento horizontal aleatório só para não empilhar.
rng = np.random.default_rng(8)

fig, ax = plt.subplots()
for posicao, categoria in enumerate(["não", "sim"]):
    valores = credit.loc[credit["estudante"] == categoria, "saldo"]
    deslocamento = rng.uniform(-0.15, 0.15, size=len(valores))
    ax.scatter(posicao + deslocamento, valores, s=12, alpha=0.4, color="C0")
ax.set_xticks([0, 1])
ax.set_xticklabels(["não", "sim"])
ax.set_xlabel("estudante")
ax.set_ylabel("saldo (dólares)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.random.default_rng(semente)`** cria um gerador de números aleatórios com semente fixa, e **`rng.uniform(inicio, fim, size)`** sorteia `size` valores uniformes entre `inicio` e `fim`, aqui o pequeno deslocamento horizontal de cada ponto.

A nuvem dos estudantes parece mais alta, mas quanto? Uma variável indicadora responde trocando a categoria por um número.

> **🔷 Conceito**
>
> Uma **variável indicadora** (ou *dummy*) troca uma categoria por 0 ou 1: vale 1 quando a observação está no nível escolhido, 0 nos demais. Um preditor qualitativo com $L$ níveis entra no modelo como $L - 1$ indicadoras, nunca $L$. A categoria que sobra sem indicadora própria é a **base**. Quando o preditor qualitativo é o único do modelo, toda previsão para a base sai só do intercepto; com outros preditores no modelo, o intercepto passa a ser o valor da base quando os demais valem zero.

In [ ]:
indicadora_estudante = pd.get_dummies(credit[["estudante"]], drop_first=True)
indicadora_estudante.columns.tolist()

> **🔧 Função**
>
> **`pd.get_dummies(df, drop_first)`**: troca cada coluna categórica de `df` por uma coluna indicadora por categoria, com nome `coluna_categoria` e valores `True`/`False`.
>
> - `drop_first=True`: descarta a indicadora da primeira categoria, que vira a base.

`drop_first=True` descarta a primeira categoria em ordem alfabética (`não` vem antes de `sim`), e fica só `estudante_sim`, que vale `True` para quem é estudante e `False` para quem não é. `não` é a base: toda observação sem a indicadora marcada pertence a ela.

In [ ]:
modelo_estudante = LinearRegression().fit(indicadora_estudante, credit["saldo"])
intercepto_estudante = float(modelo_estudante.intercept_)
coef_estudante_sim = float(modelo_estudante.coef_[0])

round(intercepto_estudante, 3), round(coef_estudante_sim, 3)

In [ ]:
medias_estudante = credit.groupby("estudante")["saldo"].mean()
media_nao_estudante = float(medias_estudante["não"])
media_sim_estudante = float(medias_estudante["sim"])
diferenca_medias_estudante = media_sim_estudante - media_nao_estudante
estudantes_devem_mais = bool(diferenca_medias_estudante > 0)

(
    round(media_nao_estudante, 3),
    round(media_sim_estudante, 3),
    round(diferenca_medias_estudante, 3),
    estudantes_devem_mais,
)

In [ ]:
intercepto_bate_com_media = bool(np.isclose(intercepto_estudante, media_nao_estudante))
coeficiente_bate_com_diferenca = bool(
    np.isclose(coef_estudante_sim, diferenca_medias_estudante)
)

intercepto_bate_com_media, coeficiente_bate_com_diferenca

O modelo ajustado só com essa indicadora dá um intercepto de 480,369 e um coeficiente de 396,456. São os mesmos dois números que saem direto do dado, sem ajustar modelo nenhum. 480,369 é o saldo médio de quem não é estudante, e 396,456 é a diferença entre essa média e a de quem é estudante, 876,825 (876,825 − 480,369 = 396,456). `intercepto_bate_com_media` e `coeficiente_bate_com_diferenca` confirmam a igualdade nos dois casos, e `estudantes_devem_mais` confirma que quem é estudante carrega, em média, mais saldo que quem não é.

### A base é uma escolha, e a previsão não sabe disso

Qual categoria fica sem indicadora, a base, é uma escolha de quem ajusta o modelo, e não um fato sobre o dado. Com `drop_first=True`, `get_dummies` descarta a primeira categoria da ordem das categorias, que para texto é a alfabética; para inverter, basta listar as categorias na ordem desejada antes de gerar a indicadora.

In [ ]:
estudante_ordenado = pd.Categorical(credit["estudante"], categories=["sim", "não"])
indicadora_estudante_b = pd.get_dummies(
    pd.DataFrame({"estudante": estudante_ordenado}), drop_first=True
)
indicadora_estudante_b.columns.tolist()

> **🔧 Função**
>
> **`pd.Categorical(valores, categories)`**: guarda os valores como categorias numa ordem fixada por `categories`, em vez da ordem alfabética. É essa ordem que `get_dummies` usa para decidir qual categoria é a primeira.

In [ ]:
modelo_estudante_b = LinearRegression().fit(indicadora_estudante_b, credit["saldo"])
intercepto_estudante_b = float(modelo_estudante_b.intercept_)
coef_estudante_nao = float(modelo_estudante_b.coef_[0])

round(intercepto_estudante_b, 3), round(coef_estudante_nao, 3)

Com `sim` como base, o intercepto salta para 876,825, a média de quem é estudante, e o coeficiente muda de sinal, para -396,456: a mesma diferença de antes, contada na direção oposta.

In [ ]:
previsao_original = modelo_estudante.predict(indicadora_estudante)
previsao_invertida = modelo_estudante_b.predict(indicadora_estudante_b)
previsoes_identicas = bool(np.allclose(previsao_original, previsao_invertida))

previsoes_identicas

`previsoes_identicas` confirma que, apesar de o intercepto e o coeficiente mudarem, as previsões para os 400 clientes são as mesmas nos dois ajustes. A escolha da base muda a leitura dos coeficientes, não o que o modelo prevê.

### Região: quando a categoria tem mais de dois nomes

`regiao` tem três categorias, `Leste`, `Sul` e `Oeste`. Uma única indicadora não dá conta, porque sobraria uma categoria sem representação. A regra do conceito acima vale de novo, com $L = 3$: entram $L - 1 = 2$ indicadoras.

In [ ]:
indicadora_regiao = pd.get_dummies(credit[["regiao"]], drop_first=True)
indicadora_regiao.columns.tolist()

Duas indicadoras, `regiao_Oeste` e `regiao_Sul`; `Leste` fica sem indicadora própria e é a base. Uma terceira indicadora, para `Leste`, seria redundante: quem não é `Oeste` nem `Sul` só pode ser `Leste`, e o valor dela já está determinado pelas outras duas. Antes de ajustar: se o intercepto vai ser o saldo médio do Leste, o que os dois coeficientes devem medir?

In [ ]:
modelo_regiao = LinearRegression().fit(indicadora_regiao, credit["saldo"])
coeficientes_regiao = dict(zip(modelo_regiao.feature_names_in_, modelo_regiao.coef_))
coef_oeste = round(float(coeficientes_regiao["regiao_Oeste"]), 2)
coef_sul = round(float(coeficientes_regiao["regiao_Sul"]), 2)
intercepto_regiao = round(float(modelo_regiao.intercept_), 2)

intercepto_regiao, coef_oeste, coef_sul

In [ ]:
medias_regiao = credit.groupby("regiao")["saldo"].mean()
media_leste = round(float(medias_regiao["Leste"]), 2)
media_oeste = round(float(medias_regiao["Oeste"]), 2)
media_sul = round(float(medias_regiao["Sul"]), 2)
leste_tem_a_maior_media = bool(medias_regiao.idxmax() == "Leste")
maior_diferenca_entre_regioes = float(medias_regiao.max() - medias_regiao.min())
desvio_saldo = float(credit["saldo"].std())

(
    media_leste,
    media_oeste,
    media_sul,
    leste_tem_a_maior_media,
    round(maior_diferenca_entre_regioes, 2),
    round(desvio_saldo, 2),
)

> **🔧 Função**
>
> **`serie.idxmax()`**: o rótulo da posição onde está o maior valor da série. Aqui, o nome da região com o maior saldo médio.

O intercepto, 531,0, é o saldo médio de quem mora no Leste, a base. Os coeficientes se leem como diferenças contra essa base: -18,69 para `Oeste` e -12,5 para `Sul`. É o que a tabela de médias mostra: 512,31 − 531,0 = -18,69 e 518,5 − 531,0 = -12,5. `leste_tem_a_maior_media` confirma que, das três regiões, é no Leste que o saldo médio é o mais alto. Mas a maior diferença entre as médias das regiões é de 18,69 dólares, num saldo cujo desvio padrão é de 459,76. Os coeficientes de região existem e são lidos do mesmo jeito que o de `estudante`, mas medem uma diferença quase nula: saber a região de um cliente quase não muda o saldo que se espera dele.

### O que o modelo prevê: a média do grupo, e só ela

Com um único preditor qualitativo, o modelo não tem outra informação para usar: toda previsão é a média do grupo a que a observação pertence, e nada mais fino que isso. Para `estudante`, só existem duas previsões possíveis, 480,369 e 876,825; para `regiao`, só três, 531,0, 512,31 e 518,5.

In [ ]:
# Figura: Saldo de cada um dos 400 clientes, por estudante (esquerda) e por região (direita). Os pontos têm um deslocamento horizontal aleatório só para não empilhar; o traço laranja marca a média de cada grupo, a mesma que o intercepto e os coeficientes reproduzem.
rng = np.random.default_rng(8)

fig, (ax_estudante, ax_regiao) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)

paineis = [
    (ax_estudante, "estudante", ["não", "sim"], medias_estudante, "estudante"),
    (ax_regiao, "regiao", ["Leste", "Sul", "Oeste"], medias_regiao, "região"),
]
for eixo, coluna, ordem, medias, rotulo in paineis:
    for posicao, categoria in enumerate(ordem):
        valores = credit.loc[credit[coluna] == categoria, "saldo"]
        deslocamento = rng.uniform(-0.15, 0.15, size=len(valores))
        eixo.scatter(posicao + deslocamento, valores, s=12, alpha=0.4, color="C0")
        eixo.plot(
            [posicao - 0.22, posicao + 0.22],
            [medias[categoria]] * 2,
            color="C1",
            linewidth=3,
        )
    eixo.set_xticks(range(len(ordem)))
    eixo.set_xticklabels(ordem)
    eixo.set_xlabel(rotulo)

ax_estudante.set_ylabel("saldo (dólares)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`plt.subplots(1, 2, sharey=True)`**: dois painéis lado a lado com o mesmo eixo vertical, para que as alturas se comparem direto de um painel para o outro.

O traço laranja fica na mesma altura para todo ponto do grupo: é essa reta plana, por categoria, que um preditor qualitativo sozinho consegue desenhar.

## Interação e Termos Não Lineares

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.2 de James et al. (2023).

Ler um coeficiente como "o efeito do preditor, mantendo os demais fixos" pressupõe duas coisas. A primeira é que o efeito de um preditor não muda com o nível dos outros. A segunda é que esse efeito é o mesmo ao longo de toda a faixa do preditor, ou seja, que a relação é uma reta. As duas suposições se chamam **aditividade** e **linearidade**, e as duas podem ser testadas em vez de assumidas: a aditividade entre `tv` e `radio`, nos dados `Advertising`, e a linearidade entre `potencia` e `milhas_por_galao`, nos dados `Auto`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("estilo-figuras.mplstyle")

### O modelo aditivo assume demais

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
y_vendas = propaganda["vendas"]

X_aditivo = propaganda[["tv", "radio"]]
modelo_aditivo = LinearRegression().fit(X_aditivo, y_vendas)
r2_aditivo = float(r2_score(y_vendas, modelo_aditivo.predict(X_aditivo)))

round(r2_aditivo, 4)

O modelo aditivo de `tv` e `radio` explica 0,8972 da variância de vendas, e deixa no resíduo o padrão em U da seção 8.3. "Aditivo" nomeia uma suposição específica: o quanto `vendas` sobe para cada mil dólares a mais em `tv` é sempre o mesmo $\hat\beta_1$, não importa quanto se gasta em `radio`. As duas mídias contribuem em paralelo, sem uma alterar o efeito da outra. É uma suposição conveniente, e nada garante que seja verdadeira.

### Uma interação entre as duas mídias

> **🔷 Conceito**
>
> Um **termo de interação** soma ao modelo o produto de dois preditores, $X_1 \cdot X_2$. Ele relaxa a aditividade: reescrevendo $\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_1 X_2$ em função de $X_1$, o coeficiente que multiplica $X_1$ deixa de ser a constante $\beta_1$ e passa a ser $\beta_1 + \beta_3 X_2$, um valor que muda com $X_2$. O efeito de um preditor passa a depender do nível do outro.

In [ ]:
propaganda["tv_radio"] = propaganda["tv"] * propaganda["radio"]

X_interacao = propaganda[["tv", "radio", "tv_radio"]]
modelo_interacao = LinearRegression().fit(X_interacao, y_vendas)
r2_interacao = float(r2_score(y_vendas, modelo_interacao.predict(X_interacao)))

round(r2_interacao, 4)

In [ ]:
comparacao_interacao = pd.DataFrame(
    {"R²": [r2_aditivo, r2_interacao]},
    index=["aditivo (tv + radio)", "com interação (tv + radio + tv×radio)"],
).round(4)
comparacao_interacao

In [ ]:
pct_restante_explicado = round((r2_interacao - r2_aditivo) / (1 - r2_aditivo) * 100, 1)
pct_restante_explicado

Somar a coluna `tv_radio` sobe o R² de 0,8972 para 0,9678. Que ele não caísse era garantido: o modelo com interação contém o aditivo, e o R² de treino nunca cai quando se acrescenta uma coluna. O que diz algo é o tamanho do ganho. Da variância que ainda sobrava depois do modelo aditivo, 68,7% foi explicada pela interação.

E o padrão em U que o modelo aditivo deixava no resíduo? A mesma conta da seção 8.3, a média do resíduo em cada terço do valor previsto, agora para os dois modelos:

In [ ]:
def media_do_residuo_por_terco(previsto):
    terco = pd.qcut(previsto, 3, labels=["baixo", "medio", "alto"])
    return (y_vendas - previsto).groupby(terco, observed=True).mean()

previsto_aditivo = modelo_aditivo.predict(X_aditivo)
previsto_interacao = modelo_interacao.predict(X_interacao)

residuo_por_terco = pd.DataFrame({
    "aditivo": media_do_residuo_por_terco(previsto_aditivo),
    "com interação": media_do_residuo_por_terco(previsto_interacao),
}).round(4)
residuo_por_terco

In [ ]:
todas_encolheram = bool(
    (
        residuo_por_terco["com interação"].abs() < residuo_por_terco["aditivo"].abs()
    ).all()
)
todas_encolheram

O resíduo é observado menos previsto. O modelo aditivo tem resíduo médio negativo no terço do meio (-0,8077), isto é, prevê mais do que se vendeu, e positivo nas pontas (0,4307 e 0,3650), onde prevê menos. Com a interação, as médias passam a -0,2051, 0,2926 e -0,0831, e `todas_encolheram` confirma que cada uma ficou menor em módulo que a do modelo aditivo no mesmo terço. O sinal ainda alterna, mas o erro sistemático que sobra é bem menor: a interação era a maior parte do que faltava ao plano.

In [ ]:
coef_interacao = pd.Series(
    modelo_interacao.coef_, index=modelo_interacao.feature_names_in_
)
coef_interacao_mil = (coef_interacao * 1000).round(2)

coef_interacao.round(4), coef_interacao_mil

Os três coeficientes ficam positivos: 0,0191 para `tv`, 0,0289 para `radio`, 0,0011 para `tv_radio`. Multiplicados por mil, em unidades vendidas, eles valem 19,10, 28,86 e 1,09, e compõem a leitura que o conceito acima descreveu em símbolos. Cada mil dólares a mais em `tv`, mantendo `radio` fixo, está associado a (19,10 + 1,09 × radio) unidades a mais de venda, com `radio` em milhares de dólares. Cada mil dólares a mais em `radio`, mantendo `tv` fixo, está associado a (28,86 + 1,09 × tv) unidades a mais. O efeito de uma mídia cresce com o nível da outra. É essa a "sinergia" que o termo de interação captura, e que falta ao modelo aditivo.

> **🔷 Conceito**
>
> O **princípio da hierarquia** diz que, se um termo de interação entra no modelo, os dois termos principais que o compõem entram juntos, mesmo que o coeficiente de algum deles pareça pequeno diante do outro. A razão é que `tv_radio` está correlacionada com `tv` e com `radio`, e excluir um dos dois muda o que a interação está de fato medindo. `tv` e `radio` continuam no modelo `tv + radio + tv_radio` acima por essa regra, não por terem coeficiente grande.

### Interação com um preditor qualitativo

A interação não pede que os dois preditores sejam numéricos. Em `Credit`, com a indicadora de `estudante` da seção 8.4, cabe a mesma pergunta entre uma variável numérica e uma categórica: o efeito de `renda` sobre `saldo` é o mesmo para quem é estudante e para quem não é?

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
indicadora_estudante = pd.get_dummies(credit[["estudante"]], drop_first=True)
credit["estudante_sim"] = indicadora_estudante["estudante_sim"].astype(int)
y_saldo = credit["saldo"]

credit.shape

In [ ]:
X_sem_interacao = credit[["renda", "estudante_sim"]]
modelo_credit_sem = LinearRegression().fit(X_sem_interacao, y_saldo)
coef_credit_sem = pd.Series(
    modelo_credit_sem.coef_, index=modelo_credit_sem.feature_names_in_
)

coef_credit_sem.round(2), round(float(modelo_credit_sem.intercept_), 2)

In [ ]:
credit["renda_estudante"] = credit["renda"] * credit["estudante_sim"]

X_com_interacao = credit[["renda", "estudante_sim", "renda_estudante"]]
modelo_credit_com = LinearRegression().fit(X_com_interacao, y_saldo)
coef_credit_com = pd.Series(
    modelo_credit_com.coef_, index=modelo_credit_com.feature_names_in_
)
inclinacao_estudante_com_interacao = round(
    float(coef_credit_com["renda"] + coef_credit_com["renda_estudante"]), 2
)

(
    coef_credit_com.round(2),
    round(float(modelo_credit_com.intercept_), 2),
    inclinacao_estudante_com_interacao,
)

Sem interação, `renda` vale 5,98 (dólares de saldo a mais por mil dólares de renda, mantendo o status de estudante fixo) e `estudante_sim` vale 382,67: a mesma diferença entre estudantes e não estudantes, não importa a renda. Com a interação, `renda` sobe para 6,22, `estudante_sim` para 476,68, e `renda_estudante` sai negativo, -2,00. A inclinação de quem é estudante passa a ser a soma 6,22 + (-2,00) = 4,22, mais baixa que a dos não estudantes. É a leitura que o modelo sem interação não permitia: ali as duas retas tinham a mesma inclinação por construção, e aqui não têm mais.

In [ ]:
# Figura: Saldo contra renda em Credit, para não estudantes (azul) e estudantes (laranja). Esquerda: modelo sem interação, com as duas retas paralelas. Direita: modelo com interação renda×estudante, com inclinações diferentes. Os pontos são os 400 clientes observados.
grade_renda = np.linspace(credit["renda"].min(), credit["renda"].max(), 100)
grupos = [(0, "C0", "não"), (1, "C1", "sim")]

fig, (ax_sem, ax_com) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)

for valor_grupo, cor, rotulo in grupos:
    pontos = credit[credit["estudante_sim"] == valor_grupo]
    ax_sem.scatter(pontos["renda"], pontos["saldo"], s=10, alpha=0.35, color=cor)
    ax_com.scatter(pontos["renda"], pontos["saldo"], s=10, alpha=0.35, color=cor)

    grade_sem = pd.DataFrame({"renda": grade_renda, "estudante_sim": valor_grupo})
    ax_sem.plot(
        grade_renda,
        modelo_credit_sem.predict(grade_sem),
        color=cor,
        linewidth=2.4,
        label=rotulo,
    )

    grade_com = pd.DataFrame(
        {
            "renda": grade_renda,
            "estudante_sim": valor_grupo,
            "renda_estudante": grade_renda * valor_grupo,
        }
    )
    ax_com.plot(
        grade_renda,
        modelo_credit_com.predict(grade_com),
        color=cor,
        linewidth=2.4,
        label=rotulo,
    )

ax_sem.set_title("sem interação")
ax_com.set_title("com interação")
ax_sem.set_xlabel("renda (milhares de dólares)")
ax_com.set_xlabel("renda (milhares de dólares)")
ax_sem.set_ylabel("saldo (dólares)")
ax_sem.legend(title="estudante")
ax_com.legend(title="estudante")
plt.tight_layout()
plt.show()

O painel esquerdo mostra o que a suposição sem interação força: duas retas com a mesma inclinação, deslocadas por 382,67 em saldo, em qualquer renda. O painel direito mostra o que a interação libera. A reta de quem é estudante nasce 476,68 acima da de quem não é, em `renda` zero, mas sobe mais devagar, e a distância entre as duas encolhe à medida que a renda cresce.

### Potência não anda em linha reta com milhas por galão

`Auto` traz uma armadilha de tipo antes de qualquer regressão: a coluna `potencia`, que deveria ser número, chega inteira como texto por causa de cinco linhas.

In [ ]:
auto = pd.read_csv("dados/Auto.csv")
tipo_de_potencia_na_leitura = str(auto["potencia"].dtype)
n_auto_bruto = len(auto)
n_potencia_interrogacao = int((auto["potencia"] == "?").sum())

auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)
n_auto_limpo = len(auto)
potencia_maxima = float(auto["potencia"].max())

(
    tipo_de_potencia_na_leitura,
    n_auto_bruto,
    n_potencia_interrogacao,
    n_auto_limpo,
    potencia_maxima,
)

São 397 carros no arquivo, e cinco trazem `potencia` como o texto `"?"` em vez de um número. É por isso que a coluna chega do `read_csv` com tipo `object`, de texto, e não como `float`. `pd.to_numeric(..., errors="coerce")` troca cada `"?"` por `NaN` em vez de derrubar a leitura inteira, e `dropna` descarta essas cinco linhas: restam 392 carros para o que segue, com `potencia` chegando a 230 hp. `reset_index(drop=True)` renumera as linhas de 0 a 391, para que as posições voltem a ser contínuas depois do descarte.

In [ ]:
X_potencia = auto[["potencia"]]
y_milhas = auto["milhas_por_galao"]

modelos_grau = {}
r2_grau = {}
for grau in (1, 2, 5):
    modelo = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=grau, include_bias=False),
        LinearRegression(),
    )
    modelo.fit(X_potencia, y_milhas)
    modelos_grau[grau] = modelo
    r2_grau[grau] = float(r2_score(y_milhas, modelo.predict(X_potencia)))

comparacao_graus = pd.DataFrame(
    {"R²": [r2_grau[1], r2_grau[2], r2_grau[5]]},
    index=["grau 1 (reta)", "grau 2", "grau 5"],
).round(4)
comparacao_graus

> **🔧 Função**
>
> **`PolynomialFeatures(degree, include_bias)`**: troca cada coluna $x$ pelas suas potências $x, x^2, \dots, x^{\text{degree}}$.
>
> - `include_bias=False`: não acrescenta a coluna de 1, porque `LinearRegression` já ajusta o intercepto.
>
> **`StandardScaler()`**: subtrai de cada coluna a sua média e divide pelo seu desvio padrão, deixando-a com média 0 e desvio 1.
>
> **`make_pipeline(passo1, passo2, ..., estimador)`**: encadeia transformações e um estimador num objeto só. `fit` aplica cada passo em ordem e ajusta o estimador no fim; `predict` repete as mesmas transformações antes de prever, sem que seja preciso lembrá-las.

`StandardScaler` entra antes de `PolynomialFeatures` no `Pipeline` porque `potencia` chega a 230. A quinta potência de um número desse tamanho deixa as colunas do modelo em escalas tão distintas que a solução de mínimos quadrados perde precisão. Centralizar e normalizar antes de elevar à potência evita o problema, sem mudar a curva que sai no fim.

In [ ]:
melhora_grau2_sobre_grau1 = round(r2_grau[2] - r2_grau[1], 4)
melhora_grau5_sobre_grau2 = round(r2_grau[5] - r2_grau[2], 4)
melhora_grau2_sobre_grau1, melhora_grau5_sobre_grau2

A reta explica 0,6059 da variância de milhas por galão, e a parábola de grau 2 sobe para 0,6876, uma melhora de 0,0816. Do grau 2 para o grau 5, o R² sobe só até 0,6967, uma melhora de 0,0092. (Subtraindo os valores de quatro casas da tabela, as melhoras dão 0,0817 e 0,0091; o chunk subtrai os R² completos e só arredonda no fim.) Três parâmetros a mais quase não mudam o quanto o modelo explica dos 392 carros que ele já viu. Vale a pena trocar a parábola pelo polinômio de grau 5, então? O R² sozinho não responde; a figura ajuda.

In [ ]:
# Figura: milhas_por_galao contra potencia em Auto, com a reta (grau 1), a parábola (grau 2) e o polinômio de grau 5 ajustados sobre os 392 carros com potencia numérica. Na ponta direita da faixa, as curvas de grau 2 e de grau 5 voltam a subir, a de grau 5 com mais força.
grade_potencia = pd.DataFrame(
    {
        "potencia": np.linspace(
            X_potencia["potencia"].min(), X_potencia["potencia"].max(), 300
        )
    }
)
cores_grau = {1: "C1", 2: "C2", 5: "C3"}
rotulos_grau = {1: "grau 1", 2: "grau 2", 5: "grau 5"}

fig, ax = plt.subplots()
ax.scatter(auto["potencia"], auto["milhas_por_galao"], s=14, alpha=0.35, color="C0")
for grau in (1, 2, 5):
    previsao = modelos_grau[grau].predict(grade_potencia)
    ax.plot(
        grade_potencia["potencia"],
        previsao,
        color=cores_grau[grau],
        linewidth=2.2,
        label=rotulos_grau[grau],
    )

ax.set_xlabel("potência (hp)")
ax.set_ylabel("milhas por galão")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
previsao_grau2 = modelos_grau[2].predict(grade_potencia)
previsao_grau5 = modelos_grau[5].predict(grade_potencia)
depois_de_150 = (grade_potencia["potencia"] > 150).to_numpy()

indice_minimo_grau2 = np.argmin(np.where(depois_de_150, previsao_grau2, np.inf))
indice_minimo_grau5 = np.argmin(np.where(depois_de_150, previsao_grau5, np.inf))

subida_grau2 = float(previsao_grau2[-1] - previsao_grau2[indice_minimo_grau2])
subida_grau5 = float(previsao_grau5[-1] - previsao_grau5[indice_minimo_grau5])
razao_subidas = subida_grau5 / subida_grau2

round(subida_grau2, 2), round(subida_grau5, 2), round(razao_subidas, 2)

> **🔧 Função**
>
> **`np.where(condicao, a, b)`**: elemento a elemento, pega o valor de `a` onde `condicao` é `True` e o de `b` onde é `False`. Aqui, troca por infinito as previsões abaixo de 150 hp.
>
> **`np.argmin(arr)`**: a posição do menor valor de `arr`. Com as previsões abaixo de 150 hp trocadas por infinito, encontra o ponto mais baixo da curva acima de 150 hp.

O chunk mede a subida que a figura mostra. A partir do ponto mais baixo de cada curva acima de 150 hp, a de grau 2 sobe 2,03 milhas por galão até a ponta direita, e a de grau 5 sobe 5,19, 2,56 vezes a subida da de grau 2. As duas se dobram para cima, mas a de grau 5, muito mais, e esse é um custo que o R² sozinho não mostra. Um ganho de 0,0092 no ajuste veio acompanhado de uma curva que, na ponta direita, contraria com força a tendência de queda que os 392 pontos sugerem.

### Ainda é regressão linear

Os três modelos de `potencia` acima (reta, parábola e grau 5) saíram do mesmo estimador que abriu o capítulo: `LinearRegression`, chamado dentro de um `Pipeline` que só troca as colunas de entrada, nunca o estimador. Um modelo com o quadrado e a quinta potência de `potencia` continua sendo **regressão linear**, porque "linear" descreve como o modelo combina seus parâmetros $\beta$: cada um multiplica uma coluna, e os produtos se somam. A palavra não fala do formato da curva, que pode dobrar quando a coluna multiplicada é ela mesma uma potência do preditor original. A curva dobra; a soma por trás dela continua linear nos $\beta$.

## *Outliers*, Alavancagem e Colinearidade

> **📌 Nota**
>
> Esta seção corresponde à seção 3.3.3 de James et al. (2023).

Um R² diz quanto o modelo explica, mas não diz o que está errado com ele. O erro que sobra pode ainda ter um padrão, um único carro pode estar puxando o ajuste sozinho, ou dois preditores podem andar tão juntos que seus coeficientes deixam de significar algo isolado. São quatro problemas diferentes (não linearidade, *outlier*, alavancagem e colinearidade), e cada um tem um instrumento próprio que o revela, nos dados `Auto` e `Credit`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plt.style.use("estilo-figuras.mplstyle")

auto = pd.read_csv("dados/Auto.csv")
auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)
auto["potencia2"] = auto["potencia"] ** 2

### O resíduo contra o previsto: onde a reta ainda erra

In [ ]:
y_milhas = auto["milhas_por_galao"].to_numpy()
n = len(auto)

X1 = auto[["potencia"]]
modelo1 = LinearRegression().fit(X1, y_milhas)
previsto1 = modelo1.predict(X1)
residuo1 = y_milhas - previsto1
rse1 = float(np.sqrt((residuo1**2).sum() / (n - 1 - 1)))
r2_1 = float(r2_score(y_milhas, previsto1))

X2 = auto[["potencia", "potencia2"]]
modelo2 = LinearRegression().fit(X2, y_milhas)
previsto2 = modelo2.predict(X2)
residuo2 = y_milhas - previsto2
rse2 = float(np.sqrt((residuo2**2).sum() / (n - 2 - 1)))
r2_2 = float(r2_score(y_milhas, previsto2))

n, round(r2_1, 2), round(rse1, 2), round(r2_2, 2), round(rse2, 2)

São os 392 carros de `Auto` com `potencia` numérica. Ajustada só contra `potencia`, a reta explica 0,61 da variância de `milhas_por_galao`, com erro típico de 4,91 milhas por galão. Somando a coluna `potencia2`, o quadrado de `potencia`, o R² sobe para 0,69 e o erro cai para 4,37. Essas duas medidas dizem o quanto o modelo erra em média, e não dizem se o erro que sobra tem um padrão que a reta deveria ter capturado. Para isso serve o gráfico do resíduo contra o valor previsto.

In [ ]:
# Figura: Resíduo contra o valor previsto, para o ajuste de milhas_por_galao sobre potencia (esquerda) e sobre potencia e potencia² (direita), nos 392 carros com potencia numérica. Os carros foram ordenados pelo valor previsto e divididos em oito grupos do mesmo tamanho; a linha laranja liga a média do resíduo em cada grupo. À esquerda ela desce e sobe de novo, desenhando o U que indica não linearidade; à direita as médias ficam bem mais perto de zero, com uma queda isolada no meio, sem o U pronunciado.
def medias_por_faixa(previsto, residuo, n_faixas=8):
    df = pd.DataFrame({"previsto": previsto, "residuo": residuo})
    posicao = df["previsto"].rank(method="first")
    df["faixa"] = pd.qcut(posicao, n_faixas, labels=False)
    return df.groupby("faixa").agg(
        previsto_medio=("previsto", "mean"), residuo_medio=("residuo", "mean")
    )

medias1 = medias_por_faixa(previsto1, residuo1)
medias2 = medias_por_faixa(previsto2, residuo2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
for ax, previsto, residuo, medias, titulo in [
    (ax1, previsto1, residuo1, medias1, "grau 1"),
    (ax2, previsto2, residuo2, medias2, "grau 2"),
]:
    ax.axhline(0, color="C2", linewidth=1, linestyle="--")
    ax.scatter(previsto, residuo, color="C0", s=14, alpha=0.4)
    ax.plot(
        medias["previsto_medio"], medias["residuo_medio"],
        color="C1", linewidth=2, marker="o", markersize=6,
    )
    ax.set_title(titulo)
    ax.set_xlabel("previsto (milhas por galão)")
ax1.set_ylabel("resíduo")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`serie.rank(method="first")`**: a posição de cada valor na série ordenada, de 1 a $n$. Com `method="first"`, valores empatados recebem posições distintas, na ordem em que aparecem.
>
> **`df.groupby(coluna).agg(nome=(coluna_origem, funcao), ...)`**: resume cada grupo em várias colunas de uma vez. Cada argumento cria uma coluna `nome` aplicando `funcao` (aqui, `"mean"`) à `coluna_origem`.

In [ ]:
desvio_medias1 = float(medias1["residuo_medio"].std(ddof=0))
desvio_medias2 = float(medias2["residuo_medio"].std(ddof=0))

(
    medias1["residuo_medio"].round(2).tolist(),
    medias2["residuo_medio"].round(2).tolist(),
    round(float(medias2["residuo_medio"].min()), 2),
    round(float(medias2["residuo_medio"].max()), 2),
    round(desvio_medias1, 2),
    round(desvio_medias2, 2),
)

O formato conta a história. À esquerda, as oito médias desenham um U: 2,26 no primeiro grupo e 3,85 no último, com -3,02 no meio. O desvio padrão entre elas é 2,17. À direita, o termo quadrático já resolveu boa parte desse padrão: as médias ficam entre -1,61 e 0,96, e o mesmo desvio padrão cai para 0,68. Sobra menos estrutura para o modelo explicar depois de somar o termo quadrático.

### O ponto que o modelo erra sozinho: *outlier*

No painel de grau 2 da figura acima, alguns pontos ficam bem longe do resto da nuvem, para cima ou para baixo. São carros que o modelo erra de verdade, ou só a cauda natural do ruído? Para decidir, é preciso medir o resíduo numa escala comum.

> **🔷 Conceito**
>
> Um ***outlier*** é um ponto cuja resposta observada fica longe da prevista, mesmo depois de descontar a escala típica do erro. A forma mais simples de fazer esse desconto é dividir o resíduo pelo RSE do ajuste, $e_i / \text{RSE}$, que esta seção chama de **resíduo padronizado**. A versão completa, o **resíduo estudentizado**, divide cada resíduo pelo seu próprio erro-padrão estimado, que encolhe um pouco nos pontos de alavancagem alta. O corte usual, valores acima de 3 em módulo, é o do resíduo estudentizado; quando as alavancagens são pequenas, as duas versões quase coincidem, e a seção confere isso adiante.

In [ ]:
residuo_padronizado = residuo2 / rse2
idx_pior = int(np.argmax(np.abs(residuo_padronizado)))
nome_pior = auto.loc[idx_pior, "nome"]
pior_valor = float(residuo_padronizado[idx_pior])
n_acima_de_3 = int((np.abs(residuo_padronizado) > 3).sum())
segundo_pior_valor = float(np.sort(np.abs(residuo_padronizado))[-2])

nome_pior, round(pior_valor, 2), n_acima_de_3, round(segundo_pior_valor, 2)

> **🔧 Função**
>
> **`np.argmax(arr)`**: a posição do maior valor de `arr`. Com `np.abs` por dentro, a posição do maior valor em módulo.

O maior resíduo padronizado, em valor absoluto, é o do `datsun 280-zx`: 3,63, acima do corte de 3. Mas ele não está sozinho. Cinco dos 392 carros passam de 3 em módulo, e o segundo colocado, com 3,38, fica a 3,63 − 3,38 = 0,25 do primeiro. É um candidato a *outlier* entre vários, e não um ponto que se destaca de todo o resto.

In [ ]:
mascara_sem_pior = np.ones(n, dtype=bool)
mascara_sem_pior[idx_pior] = False

modelo2_sem_pior = LinearRegression().fit(
    X2[mascara_sem_pior], y_milhas[mascara_sem_pior]
)
previsto2_sem_pior = modelo2_sem_pior.predict(X2[mascara_sem_pior])
r2_2_sem_pior = float(r2_score(y_milhas[mascara_sem_pior], previsto2_sem_pior))

rse2_sem_pior = float(
    np.sqrt(
        ((y_milhas[mascara_sem_pior] - previsto2_sem_pior) ** 2).sum() / (n - 1 - 2 - 1)
    )
)
deslocamento_sem_pior = float(
    np.sqrt(np.mean((modelo2_sem_pior.predict(X2) - previsto2) ** 2))
)


(
    round(r2_2, 4),
    round(r2_2_sem_pior, 4),
    round(rse2, 3),
    round(rse2_sem_pior, 3),
    round(deslocamento_sem_pior, 3),
)

Tirar só esse carro e reajustar mexe pouco na curva. Comparando as previsões dos dois ajustes nos 392 carros, a diferença típica (a raiz da média dos quadrados) é de 0,066 milha por galão. Mexe mais nas medidas de qualidade: o R² sobe de 0,6876 para 0,6971, e o RSE cai de 4,374 para 4,304. Um único carro, entre 392, responde por uma parte visível do erro que o ajuste declara. A curva quase não sai do lugar porque o valor de `potencia` desse carro não é incomum: como a medição logo adiante mostra, a alavancagem dele fica abaixo da média das 392 observações. Um *outlier* de baixa alavancagem distorce o RSE e o R² mais do que a curva.

### Alavancagem: quando o exagero está no eixo x

O `datsun 280-zx` é incomum na resposta: consome bem menos do que a sua potência sugere. Um carro também pode ser incomum no preditor, com potência muito acima ou muito abaixo da dos outros. Um ponto assim fica numa ponta da nuvem, onde há poucos vizinhos para contrabalançá-lo. Quanto ele pesa no ajuste?

> **🔷 Conceito**
>
> A **alavancagem** $h_i$ de uma observação mede o quão incomum é o seu valor de preditor, e não o quão longe a resposta observada fica da prevista. Em regressão simples, ela tem fórmula fechada:
>
> $$
> h_i = \frac{1}{n} + \frac{(x_i - \bar{x})^2}{\displaystyle\sum_{i'=1}^{n} (x_{i'} - \bar{x})^2}.
> $$
>
> $h_i$ cresce com a distância de $x_i$ à média, está sempre entre $1/n$ e 1, e sua média sobre as $n$ observações é sempre $(p+1)/n$, com $p$ preditores.

O ajuste aqui tem dois preditores, `potencia` e `potencia2`, e aí "longe da média" precisa levar em conta os dois de uma vez, e também como eles variam juntos. A conta que faz isso é a diagonal de uma matriz, a **matriz chapéu** (*hat matrix*). Não é preciso acompanhar a álgebra: o que importa é que $h_i$ continua medindo o quão fora do padrão está a combinação de preditores da observação $i$, e que, com um preditor só, volta a ser a fórmula acima. Para quem quiser ler a fórmula, dois símbolos são novos: $X^\top$ é a transposta de $X$, a mesma tabela com linhas e colunas trocadas, e $(\cdot)^{-1}$ é a inversa de uma matriz, o análogo de dividir por um número.

$$
H = X (X^\top X)^{-1} X^\top, \qquad h_i = H_{ii},
$$

em que $X$ tem uma coluna de 1 para o intercepto e uma coluna para cada preditor, e $H_{ii}$ é o $i$-ésimo elemento da diagonal de $H$.

In [ ]:
X_design = np.column_stack([np.ones(n), X2.to_numpy()])
H = X_design @ np.linalg.inv(X_design.T @ X_design) @ X_design.T
alavancagem = np.diag(H)

p = X2.shape[1]
media_alavancagem = float(alavancagem.mean())
alavancagem_esperada = (p + 1) / n
alavancagem_bate_com_formula = bool(np.isclose(media_alavancagem, alavancagem_esperada))

(
    round(media_alavancagem, 4),
    round(alavancagem_esperada, 4),
    alavancagem_bate_com_formula,
)

> **🔧 Função**
>
> **`np.ones(n)`**: um array de `n` uns, aqui a coluna do intercepto. **`np.column_stack(lista)`** junta os arrays da lista lado a lado, como colunas de uma matriz.
>
> **`A @ B`**: o produto de matrizes. **`M.T`** é a transposta de `M`, e **`np.linalg.inv(M)`** a sua inversa. As três juntas escrevem $X (X^\top X)^{-1} X^\top$ quase como na fórmula.
>
> **`np.diag(M)`**: a diagonal de uma matriz quadrada, como array. Aqui, os $h_i$.

A média das 392 alavancagens, 0,0077, bate com $(p+1)/n$ para $p=2$ preditores, e `alavancagem_bate_com_formula` confirma `True`. Não é um acaso deste ajuste em particular: é o que a fórmula promete para qualquer regressão linear com intercepto.

Com as alavancagens em mãos, dá para calcular também o resíduo estudentizado, que divide cada resíduo por $\text{RSE}\sqrt{1 - h_i}$ em vez de só pelo RSE, e conferir se ele muda a lista de possíveis *outliers*:

In [ ]:
residuo_estudentizado = residuo2 / (rse2 * np.sqrt(1 - alavancagem))
mesmos_carros_acima_de_3 = bool(
    np.array_equal(np.abs(residuo_estudentizado) > 3, np.abs(residuo_padronizado) > 3)
)

(
    round(float(residuo_estudentizado[idx_pior]), 2),
    int((np.abs(residuo_estudentizado) > 3).sum()),
    mesmos_carros_acima_de_3,
)

O `datsun 280-zx` passa de 3,63 para 3,65, os carros acima de 3 continuam sendo 5, e `mesmos_carros_acima_de_3` confirma que são os mesmos. Com alavancagens tão pequenas, $\sqrt{1 - h_i}$ fica perto de 1 e as duas versões quase coincidem; daqui em diante, a seção segue com o resíduo padronizado.

In [ ]:
limiar_alavancagem = 3 * media_alavancagem
alta_alavancagem = alavancagem > limiar_alavancagem
e_outlier = np.abs(residuo_padronizado) > 3

n_alta_alavancagem = int(alta_alavancagem.sum())
maior_residuo_entre_alta_alavancagem = float(
    np.max(np.abs(residuo_padronizado[alta_alavancagem]))
)
maior_alavancagem_entre_outliers = float(np.max(alavancagem[e_outlier]))
nenhum_carro_e_os_dois = bool(not np.any(alta_alavancagem & e_outlier))

(
    n_alta_alavancagem,
    round(maior_residuo_entre_alta_alavancagem, 2),
    round(maior_alavancagem_entre_outliers, 4),
    nenhum_carro_e_os_dois,
)

Um critério usual para "alavancagem alta" é passar do triplo da média. Dos 15 carros que passam desse limiar, o maior resíduo padronizado em módulo é 2,76, abaixo do corte de *outlier*. Dos 5 carros com resíduo padronizado acima de 3, a maior alavancagem é 0,0076, abaixo da própria média. `nenhum_carro_e_os_dois` confirma `True`: nenhum carro deste conjunto combina os dois problemas ao mesmo tempo, e a figura a seguir mostra isso.

In [ ]:
# Figura: Resíduo padronizado contra alavancagem, para os 392 carros no ajuste com potencia e potencia². O ponto laranja é o pior resíduo (datsun 280-zx); o ponto roxo é a maior alavancagem (pontiac grand prix). São problemas diferentes, e nenhum carro combina os dois.
idx_maior_alavancagem = int(np.argmax(alavancagem))
nome_maior_alavancagem = auto.loc[idx_maior_alavancagem, "nome"]

fig, ax = plt.subplots()
ax.axhline(0, color="C2", linewidth=1, linestyle="--")
ax.scatter(alavancagem, residuo_padronizado, color="C0", s=16, alpha=0.45)
ax.scatter(
    [alavancagem[idx_pior]], [residuo_padronizado[idx_pior]], color="C1", s=55, zorder=3
)
ax.annotate(
    nome_pior,
    xy=(alavancagem[idx_pior], residuo_padronizado[idx_pior]),
    xytext=(8, 4),
    textcoords="offset points",
    fontsize=9,
)
ax.scatter(
    [alavancagem[idx_maior_alavancagem]],
    [residuo_padronizado[idx_maior_alavancagem]],
    color="C3",
    s=55,
    zorder=3,
)
ax.annotate(
    nome_maior_alavancagem,
    xy=(alavancagem[idx_maior_alavancagem], residuo_padronizado[idx_maior_alavancagem]),
    xytext=(8, 4),
    textcoords="offset points",
    fontsize=9,
)
ax.set_xlim(left=0)
ax.set_xlabel("alavancagem")
ax.set_ylabel("resíduo padronizado")
plt.tight_layout()
plt.show()

Os cinco carros com resíduo padronizado além de 3 em módulo ocupam quantos pontos distintos na figura?

In [ ]:
outliers = auto.loc[e_outlier, ["nome", "potencia", "milhas_por_galao"]].assign(
    residuo_padronizado=np.round(residuo_padronizado[e_outlier], 2)
)
outliers

In [ ]:
pontos_distintos = len(
    set(
        zip(
            np.round(alavancagem[e_outlier], 10),
            np.round(residuo_padronizado[e_outlier], 10),
        )
    )
)
pontos_distintos

> **🔧 Função**
>
> **`df.assign(nova=valores)`**: devolve uma cópia do `DataFrame` com a coluna `nova` acrescentada, sem alterar o original.

São 4 pontos, e não 5. A tabela mostra por quê: o `mercury monarch` e o `ford maverick` têm a mesma potência, 72 hp, e o mesmo consumo, 15,0 milhas por galão. Por isso têm o mesmo resíduo e a mesma alavancagem, e caem exatamente no mesmo ponto da figura.

In [ ]:
alavancagem_do_pior_residuo = float(alavancagem[idx_pior])
maior_alavancagem = float(alavancagem[idx_maior_alavancagem])
residuo_padronizado_da_maior_alavancagem = float(
    residuo_padronizado[idx_maior_alavancagem]
)
potencia_da_maior_alavancagem = float(auto.loc[idx_maior_alavancagem, "potencia"])
potencia_e_a_maior_do_conjunto = bool(
    potencia_da_maior_alavancagem == auto["potencia"].max()
)

(
    round(alavancagem_do_pior_residuo, 4),
    round(maior_alavancagem, 2),
    round(residuo_padronizado_da_maior_alavancagem, 2),
    potencia_da_maior_alavancagem,
    potencia_e_a_maior_do_conjunto,
)

O `datsun 280-zx`, o pior resíduo, tem alavancagem 0,0066, abaixo da média 0,0077. Seu problema é só em $y$: o consumo observado fica longe do previsto para a sua `potencia`. Do outro lado, o `pontiac grand prix` tem a maior alavancagem do conjunto, 0,09, e `potencia_e_a_maior_do_conjunto` confirma por quê: sua `potencia`, 230 hp, é a mais alta entre os 392 carros. Mas seu resíduo padronizado é só 0,28, longe do corte de 3. Os dois carros ilustram, cada um do seu lado, o que o chunk anterior confirmou para o conjunto inteiro.

Tirar o pior resíduo mexeu pouco na curva. E o outro carro: a maior alavancagem do conjunto desloca o ajuste sozinha?

In [ ]:
mascara_sem_alavancagem = np.ones(n, dtype=bool)
mascara_sem_alavancagem[idx_maior_alavancagem] = False

modelo2_sem_alavancagem = LinearRegression().fit(
    X2[mascara_sem_alavancagem], y_milhas[mascara_sem_alavancagem]
)
previsto2_sem_alavancagem = modelo2_sem_alavancagem.predict(X2[mascara_sem_alavancagem])
r2_2_sem_alavancagem = float(
    r2_score(y_milhas[mascara_sem_alavancagem], previsto2_sem_alavancagem)
)
deslocamento_sem_alavancagem = float(
    np.sqrt(np.mean((modelo2_sem_alavancagem.predict(X2) - previsto2) ** 2))
)
deslocamento_na_ponta = float(
    abs(
        modelo2_sem_alavancagem.predict(X2)[idx_maior_alavancagem]
        - previsto2[idx_maior_alavancagem]
    )
)

(
    round(r2_2, 4),
    round(r2_2_sem_alavancagem, 4),
    round(deslocamento_sem_alavancagem, 3),
    round(deslocamento_na_ponta, 3),
)

Reajustando sem o `pontiac grand prix`, o R² vai de 0,6876 a 0,6869, e a diferença típica entre as previsões dos dois ajustes é de 0,021 milha por galão, menor que os 0,066 da remoção do `datsun 280-zx`. Na própria ponta de 230 hp a curva se mexe mais, 0,127 milha por galão, mas ainda pouco diante de um RSE de 4,374. Uma alavancagem alta dá a uma observação a *chance* de puxar o ajuste, e não a garantia de que ela puxe. Para puxar de fato, ela precisaria também cair longe da curva, e este carro não cai.

### Colinearidade: quando dois preditores quase se confundem

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
y_saldo = credit["saldo"].to_numpy().astype(float)

correlacoes_credit = credit[["limite", "pontuacao", "idade"]].corr()
correlacoes_credit.round(3)

Em `Credit`, `limite` e `pontuacao` correlacionam a 0,997, quase 1. `limite` e `idade` correlacionam a só 0,10. Os dois primeiros sobem juntos; `idade` não tem relação visível com nenhum deles.

In [ ]:
X_idade_limite = credit[["idade", "limite"]]
modelo_idade_limite = LinearRegression().fit(X_idade_limite, y_saldo)

X_pontuacao_limite = credit[["pontuacao", "limite"]]
modelo_pontuacao_limite = LinearRegression().fit(X_pontuacao_limite, y_saldo)

coef_idade = float(modelo_idade_limite.coef_[0])
coef_limite_com_idade = float(modelo_idade_limite.coef_[1])
coef_pontuacao = float(modelo_pontuacao_limite.coef_[0])
coef_limite_com_pontuacao = float(modelo_pontuacao_limite.coef_[1])

r2_idade_limite = float(r2_score(y_saldo, modelo_idade_limite.predict(X_idade_limite)))
r2_pontuacao_limite = float(
    r2_score(y_saldo, modelo_pontuacao_limite.predict(X_pontuacao_limite))
)

(
    round(coef_idade, 2),
    round(coef_limite_com_idade, 2),
    round(coef_pontuacao, 2),
    round(coef_limite_com_pontuacao, 2),
    round(r2_idade_limite, 4),
    round(r2_pontuacao_limite, 4),
)

Antes de olhar os coeficientes: se `pontuacao` quase repete `limite`, o que deve acontecer com o coeficiente de `limite` quando os dois entram juntos?

Com `idade`, o coeficiente de `limite` é 0,17 (o de `idade`, -2,29): mais um dólar de limite está associado a 0,17 dólar a mais de saldo, mantendo a idade fixa. Trocando `idade` por `pontuacao`, que anda junto com `limite`, o coeficiente de `limite` cai para 0,02, e o de `pontuacao` sai 2,20. O modelo quase não muda de poder explicativo: o R² fica em 0,7498 com `idade` e 0,7459 com `pontuacao`. A colinearidade só redistribuiu o crédito entre os dois preditores que se sobrepõem, e não há como separar, só olhando os coeficientes, quanto é de cada um.

> **🔷 Conceito**
>
> O **fator de inflação da variância** (VIF) de um preditor $X_j$ é
>
> $$
> \text{VIF}(\hat\beta_j) = \frac{1}{1 - R^2_{X_j \mid X_{-j}}},
> $$
>
> em que $R^2_{X_j \mid X_{-j}}$ é o R² da regressão de $X_j$ sobre todos os demais preditores. VIF = 1 é ausência completa de colinearidade. Valores acima de 5 ou de 10 costumam ser tratados como colinearidade problemática.

In [ ]:
def vif(preditor, tabela, todos):
    outros = [c for c in todos if c != preditor]
    modelo_aux = LinearRegression().fit(tabela[outros], tabela[preditor])
    r2_aux = r2_score(tabela[preditor], modelo_aux.predict(tabela[outros]))
    return 1 / (1 - r2_aux)

preditores_vif = ["idade", "limite", "pontuacao"]
vif_idade = vif("idade", credit, preditores_vif)
vif_limite = vif("limite", credit, preditores_vif)
vif_pontuacao = vif("pontuacao", credit, preditores_vif)

round(vif_idade, 2), round(vif_limite, 2), round(vif_pontuacao, 2)

`idade` sai com VIF 1,01, sem colinearidade. `limite` e `pontuacao` saem com 160,59 e 160,67, muito acima da faixa de 5 a 10. A correlação já mostrava os dois andando juntos, e o VIF põe número nisso: quanto mais alto ele é, mais larga fica a faixa de pares de coeficientes que quase empatam em RSS.

RSS, como função dos coeficientes, é um vale com um único fundo, como na seção 8.1. Uma colinearidade forte, mas não perfeita, não muda isso: ainda há um único mínimo. O que ela muda é o formato do vale ao redor dele. (Com colinearidade perfeita, uma coluna repetindo exatamente a outra, o fundo viraria uma reta inteira de mínimos.)

In [ ]:
# Figura: Curvas de nível do RSS da regressão de saldo sobre dois preditores, em função dos coeficientes desses dois preditores, com o intercepto no seu melhor valor. Esquerda: idade e limite, que quase não se correlacionam; o vale é uma elipse larga. Direita: pontuacao e limite, que correlacionam a 0,997; o vale vira uma calha comprida na diagonal, e muitos pares (coeficiente de pontuacao, coeficiente de limite) quase empatam em RSS. Os eixos dos dois painéis estão em escalas diferentes, e é a medida do texto, logo abaixo, que compara as duas calhas numa régua só. No painel esquerdo, quanto mais grossa a curva, menor o RSS que ela marca; no direito, as curvas vão todas em traço fino, para não se fundirem na calha. O ponto marca o par que minimiza RSS em cada ajuste.
def grade_rss(
    preditores, k=8, pontos=700, niveis_relativos=(0.02, 0.05, 0.1, 0.2, 0.4)
):
    X = credit[preditores].to_numpy(dtype=float)
    Xc = X - X.mean(axis=0)
    yc = y_saldo - y_saldo.mean()
    M = Xc.T @ Xc
    M_inv = np.linalg.inv(M)
    beta_hat = np.linalg.lstsq(Xc, yc, rcond=None)[0]
    rss_min = float(((yc - Xc @ beta_hat) ** 2).sum())

    largura0 = np.sqrt(0.01 * rss_min * M_inv[0, 0])
    largura1 = np.sqrt(0.01 * rss_min * M_inv[1, 1])
    grade0 = np.linspace(beta_hat[0] - k * largura0, beta_hat[0] + k * largura0, pontos)
    grade1 = np.linspace(beta_hat[1] - k * largura1, beta_hat[1] + k * largura1, pontos)
    malha0, malha1 = np.meshgrid(grade0, grade1)
    d0, d1 = malha0 - beta_hat[0], malha1 - beta_hat[1]
    rss = rss_min + M[0, 0] * d0**2 + 2 * M[0, 1] * d0 * d1 + M[1, 1] * d1**2
    niveis = rss_min * (1 + np.array(niveis_relativos))
    return malha0, malha1, rss, niveis, beta_hat


larguras_das_curvas = [2.2, 1.8, 1.4, 1.0, 0.7]

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 4.6))
contornos_pares = []
for ax, preditores, rotulos, larguras in [
    (
        ax_a,
        ["idade", "limite"],
        ("coeficiente de idade", "coeficiente de limite"),
        larguras_das_curvas,
    ),
    (
        ax_b,
        ["pontuacao", "limite"],
        ("coeficiente de pontuacao", "coeficiente de limite"),
        0.7,
    ),
]:
    malha0, malha1, rss, niveis, beta_hat = grade_rss(preditores)
    contornos = ax.contour(
        malha0, malha1, rss, levels=niveis, colors="C0", linewidths=larguras
    )
    ax.scatter([beta_hat[0]], [beta_hat[1]], color="C1", s=35, zorder=3)
    ax.set_xlabel(rotulos[0])
    ax.set_ylabel(rotulos[1])
    contornos_pares.append(contornos)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`np.linalg.lstsq(X, y, rcond=None)`**: resolve mínimos quadrados direto em álgebra linear, sem criar um estimador; o primeiro elemento do que devolve são os coeficientes. Com `X` e `y` centrados na média, dá os mesmos coeficientes de `LinearRegression`, com o intercepto já descontado.

In [ ]:
def conta_fechadas(contornos):
    total, fechadas = 0, 0
    for segmentos_do_nivel in contornos.allsegs:
        for segmento in segmentos_do_nivel:
            if len(segmento) == 0:
                continue
            total += 1
            if np.allclose(segmento[0], segmento[-1]):
                fechadas += 1
    return total, fechadas

total_a, fechadas_a = conta_fechadas(contornos_pares[0])
total_b, fechadas_b = conta_fechadas(contornos_pares[1])

total_a, fechadas_a, total_b, fechadas_b

As cinco curvas de cada painel fecham: `fechadas_a` e `fechadas_b` batem com `total_a` e `total_b`, 5 de 5 nos dois casos. A janela mostra o vale inteiro nos dois pares, tanto no mais redondo quanto no esticado.

In [ ]:
def largura_beta_limite(preditores, tolerancia):
    X = credit[preditores].to_numpy(dtype=float)
    Xc = X - X.mean(axis=0)
    yc = y_saldo - y_saldo.mean()
    M = Xc.T @ Xc
    M_inv = np.linalg.inv(M)
    beta_hat = np.linalg.lstsq(Xc, yc, rcond=None)[0]
    rss_min = float(((yc - Xc @ beta_hat) ** 2).sum())
    return 2 * float(np.sqrt(tolerancia * rss_min * M_inv[1, 1]))


def razao_larguras(tolerancia):
    return largura_beta_limite(
        ["pontuacao", "limite"], tolerancia
    ) / largura_beta_limite(["idade", "limite"], tolerancia)


largura_idade_limite = largura_beta_limite(["idade", "limite"], 0.01)
largura_pontuacao_limite = largura_beta_limite(["pontuacao", "limite"], 0.01)
razao_com_1_por_cento = razao_larguras(0.01)
razao_com_5_por_cento = razao_larguras(0.05)
razao_nao_depende_da_tolerancia = bool(
    np.isclose(razao_com_1_por_cento, razao_com_5_por_cento)
)

(
    round(largura_idade_limite, 5),
    round(largura_pontuacao_limite, 5),
    round(razao_com_1_por_cento, 2),
    razao_nao_depende_da_tolerancia,
)

A medida é a largura da calha. Para cada valor de $\beta_{\text{limite}}$, o outro coeficiente é reajustado ao seu melhor valor, e a pergunta é qual faixa de $\beta_{\text{limite}}$ ainda deixa RSS a até 1% do mínimo. Essa faixa mede 0,02003 no par (idade, limite) e 0,25438 no par (pontuacao, limite): 12,7 vezes mais larga. O limiar de 1% é só uma janela para medir. As duas larguras crescem com a raiz quadrada da tolerância, que por isso cancela na divisão: repetindo a conta com 5% em vez de 1%, `razao_nao_depende_da_tolerancia` confirma `True`. Quanto mais colinear o par, mais larga é a faixa de coeficientes que o dado observado quase não distingue, e é isso que torna os coeficientes instáveis.

### Cada problema com o seu instrumento

Não linearidade, *outlier*, alavancagem e colinearidade não vêm sempre juntos. Em `Auto`, o U do resíduo tinha causa clara, a falta do termo quadrático, e diminuiu bastante com ela: o desvio padrão das médias por grupo caiu de 2,17 para 0,68. O *outlier* e a maior alavancagem recaíram sobre carros diferentes, e nenhum dos dois deslocou a curva sozinho: a diferença típica entre as previsões foi de 0,066 milha por galão sem o *outlier* e de 0,021 sem a maior alavancagem. O *outlier*, mesmo com alavancagem baixa, mexeu no R² (de 0,6876 para 0,6971) e no RSE (de 4,374 para 4,304). A colinearidade só apareceu em `Credit`, com dois preditores quase colados, e alargou em 12,7 vezes a faixa de coeficientes que quase empatam em RSS.

Cada problema pede o seu instrumento. Os três primeiros aparecem em dois gráficos: o resíduo contra o previsto, e o resíduo padronizado contra a alavancagem. A colinearidade aparece num número, o VIF. Nenhum deles aparece olhando só o R².

## Regressão Linear contra *k*-NN

> **📌 Nota**
>
> Esta seção corresponde à seção 3.5 de James et al. (2023).

A regressão linear aposta numa forma para $f$: uma soma de coeficientes, cada um multiplicando uma coluna. O *k*-NN da seção 7.3 não aposta em forma nenhuma, e deixa a vizinhança de cada ponto decidir o valor previsto ali. Quando a forma que a reta assume compensa, e quando ela custa caro? A régua é a da seção 7.6: o MSE de teste, medido contra uma $f$ simulada e conhecida.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("estilo-figuras.mplstyle")

### Quando a forma verdadeira é a da reta

Responder a pergunta com `Auto` ou `Credit` não dá. A $f$ que gerou `milhas_por_galao` ou `saldo` é desconhecida. O erro de teste diria qual método erra menos naquele dado, mas não por quê, e não há como mudar a forma da verdade de propósito para ver o que acontece. A saída é simular o próprio dado, com a semente `np.random.default_rng(8)`, começando pelo cenário mais favorável à reta: uma verdade exatamente linear.

In [ ]:
def f_verdadeiro(x, curvatura, freq=0.8):
    return 2.0 + 1.5 * x + curvatura * np.sin(freq * x)

rng = np.random.default_rng(8)
n_treino = 50
ruido_padrao = 0.5

x_treino = rng.uniform(-3, 3, size=n_treino)
ruido_treino = rng.normal(0, ruido_padrao, size=n_treino)
y_treino_linear = f_verdadeiro(x_treino, 0.0) + ruido_treino

n_treino, ruido_padrao

Cinquenta pontos de treino, $x$ uniforme entre -3 e 3, com ruído normal de desvio padrão 0,5 somado a uma reta verdadeira, $f(x) = 2 + 1{,}5x$. A função `f_verdadeiro` carrega um segundo termo, $\text{curvatura} \cdot \sin(0{,}8x)$, que fica zerado por enquanto. Mais adiante ele é ligado aos poucos, sem mudar mais nada.

In [ ]:
reta = LinearRegression().fit(x_treino.reshape(-1, 1), y_treino_linear)
knn1 = KNeighborsRegressor(n_neighbors=1).fit(x_treino.reshape(-1, 1), y_treino_linear)
knn9 = KNeighborsRegressor(n_neighbors=9).fit(x_treino.reshape(-1, 1), y_treino_linear)

mse_treino_reta = mean_squared_error(
    y_treino_linear, reta.predict(x_treino.reshape(-1, 1))
)
mse_treino_knn1 = mean_squared_error(
    y_treino_linear, knn1.predict(x_treino.reshape(-1, 1))
)
mse_treino_knn9 = mean_squared_error(
    y_treino_linear, knn9.predict(x_treino.reshape(-1, 1))
)

coef_reta = float(reta.coef_[0])
intercepto_reta = float(reta.intercept_)

(
    round(mse_treino_reta, 2),
    round(mse_treino_knn1, 2),
    round(mse_treino_knn9, 2),
    round(coef_reta, 2),
    round(intercepto_reta, 2),
)

> **🔧 Função**
>
> **`KNeighborsRegressor(n_neighbors).fit(X, y)`**: o *k*-NN para resposta numérica. Prevê, para cada ponto, a média da resposta dos `n_neighbors` pontos de treino mais próximos.
>
> **`arr.reshape(-1, 1)`**: transforma um array de uma dimensão numa coluna, com uma linha por valor; o `-1` deixa o `numpy` calcular quantas linhas. Os estimadores do `scikit-learn` esperam `X` com uma coluna por preditor, mesmo quando há um preditor só.

Sobre o próprio treino, a reta erra 0,26 de MSE. O *k*-NN com $k = 1$ erra 0,00, porque cada ponto é seu próprio vizinho mais próximo e a previsão para ele é a resposta que ele mesmo tinha. Com $k = 9$, erra 0,31, mais que a reta. O coeficiente que a reta encontrou, 1,50, praticamente repete o 1,5 verdadeiro, e o intercepto, 2,08, fica perto do 2,0 verdadeiro, com a folga vinda só do ruído desses cinquenta pontos.

In [ ]:
n_teste_grande = 20_000
x_teste_grande = rng.uniform(-3, 3, size=n_teste_grande)
ruido_teste_grande = rng.normal(0, ruido_padrao, size=n_teste_grande)
y_teste_grande_linear = f_verdadeiro(x_teste_grande, 0.0) + ruido_teste_grande

n_teste_grande

Vinte mil pontos novos, nunca usados no ajuste, servem só para medir o erro de teste com uma precisão que cinquenta pontos não entregam.

In [ ]:
ks = [1, 3, 5, 7, 9, 13, 19, 27, 39]
mse_teste_reta_linear = mean_squared_error(
    y_teste_grande_linear, reta.predict(x_teste_grande.reshape(-1, 1))
)
mses_teste_knn_linear = np.array(
    [
        mean_squared_error(
            y_teste_grande_linear,
            KNeighborsRegressor(n_neighbors=k)
            .fit(x_treino.reshape(-1, 1), y_treino_linear)
            .predict(x_teste_grande.reshape(-1, 1)),
        )
        for k in ks
    ]
)

melhor_k_linear = int(ks[int(np.argmin(mses_teste_knn_linear))])
melhor_mse_knn_linear = float(mses_teste_knn_linear.min())
pior_k_linear = int(ks[int(np.argmax(mses_teste_knn_linear))])
pior_mse_knn_linear = float(mses_teste_knn_linear.max())
reta_vence_sempre_quando_linear = bool(
    mse_teste_reta_linear < mses_teste_knn_linear.min()
)

(
    round(float(mse_teste_reta_linear), 2),
    melhor_k_linear,
    round(melhor_mse_knn_linear, 2),
    pior_k_linear,
    round(pior_mse_knn_linear, 2),
    round(float(mses_teste_knn_linear[0]), 2),
    reta_vence_sempre_quando_linear,
)

In [ ]:
# Figura: Esquerda: cinquenta pontos de treino, a f verdadeira (uma reta, pois curvatura=0), a reta ajustada (tracejada) e os ajustes de *k*-NN com $k = 1$ (linha fina) e $k = 9$ (linha grossa). Direita: MSE de teste (vinte mil pontos) contra 1/k, em escala log; a reta ajustada é a linha tracejada horizontal, porque ela não depende de k. Em nenhum dos nove valores de $k$ o *k*-NN desce abaixo dela.
grade = np.linspace(x_teste_grande.min(), x_teste_grande.max(), 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

ax1.scatter(x_treino, y_treino_linear, color="C0", s=18, alpha=0.6, label="treino")
ax1.plot(
    grade, f_verdadeiro(grade, 0.0), color="C2", linewidth=2.4, label="f verdadeira"
)
ax1.plot(
    grade,
    reta.predict(grade.reshape(-1, 1)),
    color="C1",
    linewidth=2,
    linestyle="--",
    label="reta",
)
ax1.plot(
    grade,
    knn9.predict(grade.reshape(-1, 1)),
    color="C3",
    linewidth=2.4,
    label="k-NN, k=9",
)
ax1.plot(
    grade,
    knn1.predict(grade.reshape(-1, 1)),
    color="C3",
    linewidth=0.9,
    label="k-NN, k=1",
)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.legend(loc="upper left")

inverso_k = 1 / np.array(ks)
ax2.plot(
    inverso_k,
    mses_teste_knn_linear,
    color="C3",
    linewidth=2,
    marker="o",
    markersize=4,
    label="k-NN",
)
ax2.axhline(
    mse_teste_reta_linear, color="C1", linewidth=2, linestyle="--", label="reta"
)
ax2.set_xscale("log")
ax2.set_xlim(1 / max(ks) / 1.3, 1.3)
ax2.set_xlabel("1/k (escala log)")
ax2.set_ylabel("MSE de teste")
ax2.legend()

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.set_xscale("log")`**: põe o eixo horizontal em escala logarítmica, em que cada marca vale um múltiplo fixo da anterior. Aqui, espaça por igual $1/k$ de 1/39 a 1, valores de ordens de grandeza diferentes. **`ax.set_xlim(inicio, fim)`** fixa os limites do eixo.

No teste grande, a reta erra 0,25 de MSE. O menor erro do *k*-NN, entre os nove valores de $k$ varridos, sai em $k = 7$ (`melhor_k_linear`) e chega a 0,35, acima do 0,25 da reta. `reta_vence_sempre_quando_linear` confirma `True`: nenhum dos nove valores de $k$ derruba a reta quando a verdade é dela mesma.

O contraste mais direto é com $k = 1$: 0,00 de erro no treino, 0,47 no teste. O ajuste que decorou a amostra de cinquenta pontos erra, em média, bem mais num ponto novo. É o sobreajuste da seção 7.3. O pior da varredura, porém, é o extremo oposto: $k = 39$ (`pior_k_linear`), com 3,97. Uma vizinhança que junta quase todos os cinquenta pontos de treino achata a inclinação da reta verdadeira e erra sistematicamente nas duas pontas do intervalo de $x$.

### Quando a curvatura cresce

Se a verdade se afasta de uma reta, quem deve sofrer mais: a reta, que não tem como se dobrar, ou o *k*-NN, que não assume forma nenhuma?

A verdade linear é o caso mais favorável à reta, porque nenhuma forma bate a forma certa. Dado real raramente entrega isso. O termo $\text{curvatura} \cdot \sin(0{,}8x)$ de `f_verdadeiro`, deixado em zero até aqui, agora é ligado aos poucos. Os cinquenta valores de $x$ de treino e o ruído que os acompanha ficam os mesmos, e o teste é sempre contra os mesmos vinte mil pontos, com o mesmo ruído. A única coisa que muda de uma rodada para a seguinte é o quanto a verdade se afasta de uma reta.

Para o *k*-NN, cada rodada guarda o menor MSE de teste entre os nove valores de $k$. É o melhor cenário possível para ele: o $k$ é escolhido olhando o próprio teste, o que na prática não se pode fazer, porque o teste deixaria de medir erro em dado novo. Na prática, $k$ se escolhe sem espiar o teste.

In [ ]:
curvaturas = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0])
mses_reta_curvatura = []
mses_knn_curvatura = []
for c in curvaturas:
    y_treino_c = f_verdadeiro(x_treino, c) + ruido_treino
    reta_c = LinearRegression().fit(x_treino.reshape(-1, 1), y_treino_c)
    y_teste_c = f_verdadeiro(x_teste_grande, c) + ruido_teste_grande

    mse_reta_c = mean_squared_error(
        y_teste_c, reta_c.predict(x_teste_grande.reshape(-1, 1))
    )
    mses_knn_c = [
        mean_squared_error(
            y_teste_c,
            KNeighborsRegressor(n_neighbors=k)
            .fit(x_treino.reshape(-1, 1), y_treino_c)
            .predict(x_teste_grande.reshape(-1, 1)),
        )
        for k in ks
    ]
    mses_reta_curvatura.append(mse_reta_c)
    mses_knn_curvatura.append(min(mses_knn_c))

mses_reta_curvatura = np.array(mses_reta_curvatura)
mses_knn_curvatura = np.array(mses_knn_curvatura)
knn_vence = mses_knn_curvatura < mses_reta_curvatura

indice_transicao = int(np.argmax(knn_vence))
curvatura_antes = float(curvaturas[indice_transicao - 1])
curvatura_depois = float(curvaturas[indice_transicao])
transicao_e_unica = bool(np.all(knn_vence[indice_transicao:]))

reta_piora_a_partir_do_segundo_nivel = bool(
    np.all(np.diff(mses_reta_curvatura[1:]) > 0)
)
primeiro_passo_nao_piora = bool(np.diff(mses_reta_curvatura)[0] <= 0)
amplitude_knn = float(mses_knn_curvatura.max() - mses_knn_curvatura.min())

{
    "reta_piora_a_partir_do_segundo_nivel": reta_piora_a_partir_do_segundo_nivel,
    "primeiro_passo_nao_piora": primeiro_passo_nao_piora,
    "amplitude_knn": round(amplitude_knn, 2),
    "reta em curvatura 0,0 / 3,0": (
        round(float(mses_reta_curvatura[0]), 2),
        round(float(mses_reta_curvatura[-1]), 2),
    ),
    "k-NN em curvatura 0,0 / 3,0": (
        round(float(mses_knn_curvatura[0]), 2),
        round(float(mses_knn_curvatura[-1]), 2),
    ),
    "antes da travessia (curvatura, reta, k-NN)": (
        curvatura_antes,
        round(float(mses_reta_curvatura[indice_transicao - 1]), 2),
        round(float(mses_knn_curvatura[indice_transicao - 1]), 2),
    ),
    "depois da travessia (curvatura, reta, k-NN)": (
        curvatura_depois,
        round(float(mses_reta_curvatura[indice_transicao]), 2),
        round(float(mses_knn_curvatura[indice_transicao]), 2),
    ),
    "transicao_e_unica": transicao_e_unica,
}

> **🔧 Função**
>
> **`np.diff(arr)`**: a diferença entre cada elemento e o anterior, um array com um elemento a menos. Com todas as diferenças positivas, a sequência só sobe.

In [ ]:
# Figura: MSE de teste da reta e do melhor *k*-NN de cada nível (o menor entre os nove $k$ varridos acima, escolhido no próprio teste), contra a curvatura da verdade simulada. A reta piora conforme a curvatura cresce; o *k*-NN quase não se mexe. A faixa sombreada marca o intervalo, entre as curvaturas testadas, em que a curva da reta cruza a do *k*-NN.
fig, ax = plt.subplots()
ax.axvspan(curvatura_antes, curvatura_depois, color="C2", alpha=0.15)
ax.plot(
    curvaturas,
    mses_reta_curvatura,
    color="C1",
    linewidth=2,
    marker="o",
    markersize=4,
    label="reta",
)
ax.plot(
    curvaturas,
    mses_knn_curvatura,
    color="C3",
    linewidth=2,
    marker="o",
    markersize=4,
    label="k-NN (melhor k)",
)
ax.set_xlabel("curvatura")
ax.set_ylabel("MSE de teste")
ax.legend()
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.axvspan(x0, x1, color, alpha)`**: sombreia uma faixa vertical do gráfico, entre `x0` e `x1` no eixo horizontal. `alpha` controla a transparência.

Nos onze níveis varridos, a reta piora conforme a curvatura cresce, de 0,25 em curvatura 0,0 a 0,83 em curvatura 3,0. `reta_piora_a_partir_do_segundo_nivel` confirma `True`: a partir do segundo nível, cada um erra menos que o seguinte, sem exceção. O único passo que foge disso é o primeiro, de curvatura 0,0 para 0,2, e `primeiro_passo_nao_piora` confirma `True`. Uma onda de amplitude 0,2 custa à reta tão pouco que o acaso desta amostra de treino basta para compensar.

Enquanto isso, o melhor *k*-NN de cada nível mal se mexe: 0,35 no começo, 0,36 no fim, com `amplitude_knn` de 0,02 entre o maior e o menor dos onze. A curva da reta cruza a do *k*-NN entre a curvatura 1,0, onde a reta ainda vence (0,31 contra 0,34), e a 1,3, onde o *k*-NN passa à frente (0,35 contra 0,34). `transicao_e_unica` confirma `True`: em nenhum nível mais curvo o *k*-NN perde a dianteira de volta.

Os vinte mil pontos de teste tiram o ruído da *medição*: com poucas centenas, uma margem de 0,01 entre as curvas bem no cruzamento poderia inventar ou esconder uma travessia. Mas o ponto exato do cruzamento ainda depende dos cinquenta pontos de treino, que são um sorteio só. Outra amostra de treino poderia pôr a travessia um nível antes ou depois. O que se espera em qualquer amostra é o desenho geral: a reta piora depressa com a curvatura, o *k*-NN piora muito devagar, e a partir de alguma curvatura o *k*-NN passa à frente.

### A maldição da dimensionalidade

A curvatura 2,5, um dos níveis mais altos da varredura acima e já depois da travessia, no trecho em que o *k*-NN está à frente, serve de base para a última pergunta: o que acontece quando se somam preditores que não têm relação nenhuma com a resposta?

In [ ]:
curvatura_alta = 2.5
y_treino_dim = f_verdadeiro(x_treino, curvatura_alta) + ruido_treino

ps = [1, 2, 3, 4, 6, 10, 15, 20]
p_max = max(ps)
ruido_extra_treino = rng.uniform(-3, 3, size=(n_treino, p_max - 1))
ruido_extra_teste = rng.uniform(-3, 3, size=(n_teste_grande, p_max - 1))
ruido_teste_dim = rng.normal(0, ruido_padrao, size=n_teste_grande)
y_teste_dim = f_verdadeiro(x_teste_grande, curvatura_alta) + ruido_teste_dim

mses_reta_p = []
mses_knn_p = []
for p in ps:
    if p == 1:
        X_treino_p = x_treino.reshape(-1, 1)
        X_teste_p = x_teste_grande.reshape(-1, 1)
    else:
        X_treino_p = np.column_stack([x_treino, ruido_extra_treino[:, : p - 1]])
        X_teste_p = np.column_stack([x_teste_grande, ruido_extra_teste[:, : p - 1]])

    reta_p = LinearRegression().fit(X_treino_p, y_treino_dim)
    mse_reta_p = mean_squared_error(y_teste_dim, reta_p.predict(X_teste_p))
    mses_knn_p_k = [
        mean_squared_error(
            y_teste_dim,
            KNeighborsRegressor(n_neighbors=k)
            .fit(X_treino_p, y_treino_dim)
            .predict(X_teste_p),
        )
        for k in ks
    ]
    mses_reta_p.append(mse_reta_p)
    mses_knn_p.append(min(mses_knn_p_k))

mses_reta_p = np.array(mses_reta_p)
mses_knn_p = np.array(mses_knn_p)

razao_knn = float(mses_knn_p[-1] / mses_knn_p[0])
razao_reta = float(mses_reta_p[-1] / mses_reta_p[0])
knn_degrada_muito_mais = bool(razao_knn > razao_reta)
knn_vence_em_p1 = bool(mses_knn_p[0] < mses_reta_p[0])
reta_vence_do_p2_em_diante = bool(np.all(mses_reta_p[1:] < mses_knn_p[1:]))
coef_x_em_p20 = float(reta_p.coef_[0])
maior_coef_ruido_em_p20 = float(np.abs(reta_p.coef_[1:]).max())

(
    round(float(mses_reta_p[0]), 2),
    round(float(mses_knn_p[0]), 2),
    round(float(mses_reta_p[-1]), 2),
    round(float(mses_knn_p[-1]), 2),
    round(razao_reta, 2),
    round(razao_knn, 2),
    knn_vence_em_p1,
    reta_vence_do_p2_em_diante,
    knn_degrada_muito_mais,
    round(coef_x_em_p20, 2),
    round(maior_coef_ruido_em_p20, 2),
)

In [ ]:
# Figura: MSE de teste da reta e do melhor *k*-NN contra o número de preditores p, com a mesma verdade fortemente não linear em x e p-1 preditores extras sem relação nenhuma com a resposta. Eixo y em escala log, porque as duas curvas crescem em ritmos muito diferentes. A faixa sombreada marca a passagem de p=1 para p=2, onde a vantagem do *k*-NN desaparece.
fig, ax = plt.subplots()
ax.axvspan(1, 2, color="C2", alpha=0.15)
ax.plot(
    ps, mses_reta_p, color="C1", linewidth=2, marker="o", markersize=4, label="reta"
)
ax.plot(
    ps,
    mses_knn_p,
    color="C3",
    linewidth=2,
    marker="o",
    markersize=4,
    label="k-NN (melhor k)",
)
ax.set_yscale("log")
ax.set_xlabel("número de preditores (p)")
ax.set_ylabel("MSE de teste (escala log)")
ax.legend()
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.set_yscale("log")`**: a mesma escala logarítmica de `set_xscale`, agora no eixo vertical, para que 0,36 e 13,84 caibam legíveis no mesmo gráfico.

Com um preditor só, `knn_vence_em_p1` confirma `True`: o *k*-NN erra 0,36 contra 0,67 da reta. Nesta simulação, basta somar um preditor de ruído para a ordem se inverter, e `reta_vence_do_p2_em_diante` confirma que a reta segue à frente em todos os sete valores de $p$ maiores testados. Nos vinte preditores, dezenove deles sem relação com a resposta, a reta chega a 0,99, 1,49 vezes o erro que tinha com um preditor só. O *k*-NN chega a 13,84, 38,5 vezes o que tinha, e `knn_degrada_muito_mais` confirma `True`.

A distância que o *k*-NN usa para achar o "vizinho mais próximo" soma a contribuição de toda coordenada, relevante ou não. Em vinte dimensões, os cinquenta pontos de treino que bastavam para cobrir bem um único eixo ficam espalhados demais para que algum fique de fato perto de um ponto novo: a vizinhança deixa de significar proximidade. É a **maldição da dimensionalidade**. A reta paga bem menos. Em $p = 20$, nenhum dos dezenove coeficientes inúteis passa de 0,13 em módulo, contra 2,48 do coeficiente de $x$. Somados, porém, esses coeficientes pequenos, estimados com só cinquenta pontos, custam à reta o erro 1,49 vezes maior. É um preço real, mas nada perto das 38,5 vezes que as mesmas coordenadas custam ao *k*-NN.

### Nenhum dos dois vence sozinho

As três medições deram três respostas diferentes. A reta venceu quando a verdade era dela mesma. O *k*-NN passou à frente a partir de certa curvatura, contanto que a dimensão ficasse baixa. E, nesta simulação, um único preditor extra sem relação com a resposta bastou para devolver a vantagem à reta. Em geral, quanto menos observações há por preditor, mais o método paramétrico leva vantagem. Qual dos dois usar num dado novo depende da forma verdadeira de $f$, que é justamente a informação que falta em qualquer problema real. Para escolher sem ela, e para escolher $k$ sem espiar o teste, o capítulo 10 apresenta a validação cruzada.

## Leituras adicionais

- O [site oficial de James et al. (2023)](https://www.statlearning.com), com o PDF gratuito do livro, os dados usados neste capítulo e os laboratórios em Python.
- [`LinearRegression`, na documentação do scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), a referência oficial do estimador usado do início ao fim deste capítulo.
- [`PolynomialFeatures`, na documentação do scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), a transformação que a seção 8.5 usa para ajustar um termo não linear sem sair da regressão linear.

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.